# 05 Curated Library Expansion

## Purpose

This notebook audits the foundational books that shaped Kinship Library, lets you add your own curated titles, enriches missing titles with Google Books metadata, and prepares an expanded catalogue using the same semantic model as the current app.

The notebook keeps a clear distinction between:

**API discovered books**  
Books found through the broader thematic collection process.

**Curated foundational books**  
Books intentionally selected because they are important to the conceptual direction of Kinship Library.

### Important

Run the notebook section by section. Before any API enrichment, edit the curated list in **Section 3**. That is the main place where you can add as many titles as you want.

## 1. Setup and project paths

In [8]:
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/kinship_library")
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"

APP_BOOKS_FILE = PROCESSED_DATA / "app_books.csv"
APP_EMBEDDINGS_FILE = MODELS / "app_book_embeddings.npy"

CURATED_AUDIT_FILE = PROCESSED_DATA / "curated_books_audit.csv"
CURATED_ENRICHED_FILE = PROCESSED_DATA / "curated_books_enriched.csv"
EXPANDED_BOOKS_FILE = PROCESSED_DATA / "app_books_expanded.csv"
EXPANDED_EMBEDDINGS_FILE = MODELS / "app_book_embeddings_expanded.npy"

print("Project:", PROJECT_ROOT)

ValueError: Mountpoint must not already contain files

## 2. Load the current application catalogue

In [ ]:
app_books = pd.read_csv(APP_BOOKS_FILE)
app_embeddings = np.load(APP_EMBEDDINGS_FILE)

print("Current app books:", len(app_books))
print("Embedding shape:", app_embeddings.shape)

assert len(app_books) == app_embeddings.shape[0]

Current app books: 713
Embedding shape: (713, 384)


## 3. Curated anchor collection

These books define the intellectual direction of Kinship Library. They are anchors, not automatic additions.

Later cells compare every anchor with the existing catalogue first. Only genuinely missing books should be sent to Google Books for enrichment.

The theme values below use the exact categories already used by the app.


In [ ]:
CURATED_BOOKS = [
    {"title": "Staying with the Trouble", "author": "Donna J. Haraway", "collection_theme": "humans_and_nature", "curation_note": "Foundational work for multispecies kinship and more than human worlds."},
    {"title": "Autobiography of a Yogi", "author": "Paramahansa Yogananda", "collection_theme": "yoga_philosophy", "curation_note": "Foundational spiritual autobiography."},
    {"title": "The Yoga Sutras of Patanjali", "author": "Patanjali", "collection_theme": "yoga_philosophy", "curation_note": "Foundational text for Yoga philosophy."},
    {"title": "Bhagavad Gita", "author": "", "collection_theme": "yoga_philosophy", "curation_note": "Foundational philosophical and spiritual text."},
    {"title": "One Simple Thing: A New Look at the Science of Yoga and How It Can Transform Your Life", "author": "Eddie Stern", "collection_theme": "yoga_way_of_life", "curation_note": "Explores yogic practices and their relationship to everyday life."},
    {"title": "Yoga Vedanta Sutras", "author": "Swami Sivananda", "collection_theme": "yoga_philosophy", "curation_note": "Foundational philosophical and spiritual text."},
    {"title": "Asana, Pranayama, Mudra and Bandha", "author": "Swami Satyananda Saraswati", "collection_theme": "yoga_way_of_life", "curation_note": "Embodied practical Yoga."},
    {"title": "Hatha Yoga Pradipika", "author": "Swami Satyananda Saraswati", "collection_theme": "yoga_way_of_life", "curation_note": "Embodied practical Yoga."},
    {"title": "Who Am I? (Nan Yar?)", "author": "Bhagavan Sri Ramana Maharshi", "collection_theme": "yoga_way_of_life", "curation_note": "Foundational philosophical and spiritual text."},
    {"title": "Yoga Vasistha", "author": "Vihari-Lala Mitra", "collection_theme": "yoga_philosophy", "curation_note": "Foundational text for Yoga philosophy."},
    {"title": "The Yoga Sutras of Patanjali", "author": "Shyam Ranganathan", "collection_theme": "yoga_philosophy", "curation_note": "Specific philosophical translation and interpretation."},
    {"title": "Vedas & Upanishads", "author": "", "collection_theme": "yoga_philosophy", "curation_note": "Foundational philosophical and spiritual texts."},
    {"title": "Yoga for Americans", "author": "Indra Devi", "collection_theme": "yoga_way_of_life", "curation_note": "Historical presentation of Yoga for a broad modern audience."},
    {"title": "Yoga Anticolonial Philosophy", "author": "Shyam Ranganathan", "collection_theme": "yoga_philosophy", "curation_note": "Anticolonial philosophical approach to Yoga."},
    {"title": "Breath", "author": "James Nestor", "collection_theme": "yoga_way_of_life", "curation_note": "Contemporary exploration of breathing and embodied practice."},
    {"title": "Shiva Purana", "author": "Bibek Debroy", "collection_theme": "yoga_way_of_life", "curation_note": "Devotional and philosophical literature related to Shiva traditions."},
    {"title": "Steps to Yoga and Yoga Initiation Papers", "author": "Swami Satyananda Saraswati", "collection_theme": "yoga_way_of_life", "curation_note": "Foundational philosophical and practical Yoga material."},
    {"title": "Sri Isopanishad", "author": "Bhaktivedanta Swami Prabhupada", "collection_theme": "yoga_philosophy", "curation_note": "Foundational philosophical and spiritual text."},
    {"title": "Nature's Echo", "author": "Thomas Crowther", "collection_theme": "humans_and_nature", "curation_note": "Contemporary ecological thinking about Nature and living systems."},
    {"title": "Braiding Sweetgrass", "author": "Robin Wall Kimmerer", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Indigenous knowledge, plant relationships and scientific knowledge."},
    {"title": "The Wisdom of No Escape and the Path of Loving-Kindness", "author": "Pema Chodron", "collection_theme": "yoga_way_of_life", "curation_note": "Contemplative teachings on awareness and compassion."},
    {"title": "The Tibetan Book of the Dead", "author": "Gyurme Dorje", "collection_theme": "yoga_philosophy", "curation_note": "Tibetan Buddhist philosophical and contemplative text."},
    {"title": "Eastern Body, Western Mind", "author": "Anodea Judith", "collection_theme": "yoga_philosophy", "curation_note": "Modern interpretation connecting chakra traditions with Western psychology."},
    {"title": "Ancestral Future", "author": "Ailton Krenak", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Indigenous philosophy on ancestry, futurity, Earth and human belonging."},
    {"title": "Ideas to Postpone the End of the World", "author": "Ailton Krenak", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Indigenous critique of modernity and human separation from the living world."},
    {"title": "Life Is Not Useful", "author": "Ailton Krenak", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Indigenous philosophical critique of utility, development and human exceptionalism."},
    {"title": "Ch'ixinakax utxiwa", "author": "Silvia Rivera Cusicanqui", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Andean decolonial thought and ch'ixi epistemology."},
    {"title": "Un mundo ch'ixi es posible", "author": "Silvia Rivera Cusicanqui", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Andean decolonial thought beyond colonial binaries."},
    {"title": "Sociología de la imagen", "author": "Silvia Rivera Cusicanqui", "collection_theme": "indigenous_knowledge_americas", "curation_note": "Andean approaches to knowledge, memory, visuality and decolonization."},
]

curated = pd.DataFrame(CURATED_BOOKS)

VALID_THEMES = {
    "humans_and_nature",
    "indigenous_knowledge_americas",
    "yoga_philosophy",
    "ayurveda_indian_spirituality",
    "yoga_way_of_life",
}

invalid_themes = sorted(set(curated["collection_theme"]) - VALID_THEMES)
display(curated)
print("Curated anchors:", len(curated))
print("Invalid collection themes:", invalid_themes)


,title,author,collection_theme,curation_note
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,Foundational work for multispecies kinship and...
1,Autobiography of a Yogi,Paramahansa Yogananda,yoga_philosophy,Foundational spiritual autobiography.
2,The Yoga Sutras of Patanjali,Patanjali,yoga_philosophy,Foundational text for Yoga philosophy.
3,Bhagavad Gita,,yoga_philosophy,Foundational philosophical and spiritual text.
4,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,Explores yogic practices and their relationshi...
5,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,Foundational philosophical and spiritual text.
6,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.
7,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.
8,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,Foundational philosophical and spiritual text.
9,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,Foundational text for Yoga philosophy.


Curated anchors: 29
Invalid collection themes: []


## 4. Normalize titles and audit what is already in the current library

In [ ]:
def normalize_title(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"^the\s+", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

app_books["_title_key"] = app_books["title"].apply(normalize_title)
curated["_title_key"] = curated["title"].apply(normalize_title)

existing_keys = set(app_books["_title_key"])

curated["already_in_app"] = curated["_title_key"].isin(existing_keys)

audit = curated[
    ["title", "author", "collection_theme", "curation_note", "already_in_app"]
].copy()

display(audit)
print()
print("Already in app:", int(audit["already_in_app"].sum()))
print("Missing from app:", int((~audit["already_in_app"]).sum()))

audit.to_csv(CURATED_AUDIT_FILE, index=False)
print("Audit saved:", CURATED_AUDIT_FILE)

,title,author,collection_theme,curation_note,already_in_app
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,Foundational work for multispecies kinship and...,False
1,Autobiography of a Yogi,Paramahansa Yogananda,yoga_philosophy,Foundational spiritual autobiography.,True
2,The Yoga Sutras of Patanjali,Patanjali,yoga_philosophy,Foundational text for Yoga philosophy.,True
3,Bhagavad Gita,,yoga_philosophy,Foundational philosophical and spiritual text.,True
4,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,Explores yogic practices and their relationshi...,False
5,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,Foundational philosophical and spiritual text.,False
6,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,False
7,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,False
8,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,Foundational philosophical and spiritual text.,False
9,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,Foundational text for Yoga philosophy.,False



Already in app: 5
Missing from app: 24
Audit saved: /content/drive/MyDrive/kinship_library/data/processed/curated_books_audit.csv


## 5. Existing catalogue duplicate audit

Before expanding the library, audit the current 713 app books for repeated works and repeated editions.

This section does **not delete anything**. It creates review tables.

The audit uses two levels:

**Likely duplicate work**  
Normalized titles are very similar. This can reveal records such as `Autobiography of a Yogi` and `An Autobiography of a Yogi`.

**Likely duplicate edition**  
Title is similar and author names are also similar. These are stronger candidates for consolidation.

Classical works such as the Bhagavad Gita and Yoga Sutras should be reviewed manually because different translators and commentators can represent meaningfully different editions.

In [ ]:
from difflib import SequenceMatcher

def normalize_author(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    tokens = [t for t in value.split() if t not in {"by", "swami", "sri"}]
    return " ".join(tokens)

def broad_work_key(title):
    title = normalize_title(title)

    removable = {
        "an", "a", "by", "with", "the",
    }

    tokens = [
        token for token in title.split()
        if token not in removable
    ]

    return " ".join(tokens)

duplicate_audit = app_books.copy()

duplicate_audit["work_key"] = duplicate_audit["title"].apply(broad_work_key)
duplicate_audit["author_key"] = duplicate_audit["author"].apply(normalize_author)

work_counts = (
    duplicate_audit
    .groupby("work_key")
    .size()
    .reset_index(name="records")
    .query("records > 1")
    .sort_values("records", ascending=False)
)

print("Potential repeated work groups:", len(work_counts))
display(work_counts.head(30))

Potential repeated work groups: 40


,work_key,records
27,autobiography of yogi,13
559,upanishads,12
610,yoga sutras of patanjali,8
48,bhagavad gita,7
147,ecofeminism,6
190,environmental science,5
537,traditional ecological knowledge,5
35,ayurveda,4
473,rig veda,4
377,more than human,4


## 6. Inspect the largest repeated work groups

This makes the duplicate audit human readable.

Look especially for books where many records represent essentially the same work rather than meaningful translations, commentaries or editions.

In [ ]:
TOP_DUPLICATE_GROUPS = 20

for work_key in work_counts.head(TOP_DUPLICATE_GROUPS)["work_key"]:

    group = duplicate_audit[
        duplicate_audit["work_key"] == work_key
    ].copy()

    print()
    print("=" * 80)
    print("WORK:", work_key)
    print("RECORDS:", len(group))

    columns = [
        col for col in [
            "title",
            "author",
            "collection_theme",
            "description_final",
            "isbn_clean",
            "cover_url",
        ]
        if col in group.columns
    ]

    display(group[columns])


WORK: autobiography of yogi
RECORDS: 13


,title,author,collection_theme,description_final,isbn_clean,cover_url
345,Autobiography of a Yogi,Yogananda Paramahansa,ayurveda_indian_spirituality,NaN,8172345364,https://covers.openlibrary.org/b/isbn/81723453...
346,An Autobiography of a Yogi,Paramhansa Yogananda,ayurveda_indian_spirituality,Autobiography Of A Yogi: With A Preface By W. ...,8183630693,https://covers.openlibrary.org/b/isbn/81836306...
348,Autobiography of a Yogi,Yogananda Paramahamsa,ayurveda_indian_spirituality,The life story of spiritualist Yogananda from ...,818953551X,https://covers.openlibrary.org/b/isbn/81895355...
349,Autobiography of a Yogi,Pramahansa Yogananda,ayurveda_indian_spirituality,"""As an eyewitness recountal of the extraordina...",1650455674,https://covers.openlibrary.org/b/isbn/16504556...
350,Autobiography of a YOGI,Paramahansa Yogananda,ayurveda_indian_spirituality,"""Autobiography of a Yogi"" is an autobiography ...",9783956761508,https://covers.openlibrary.org/b/isbn/97839567...
353,Autobiography of a Yogi,"Paramahansa Paramahansa Yogananda, Adriano Luc...",ayurveda_indian_spirituality,This is the 70th-anniversary edition of Autobi...,1519296606,https://covers.openlibrary.org/b/isbn/15192966...
354,Autobiography of a Yogi,PARAMAHANSA YOGANADA,ayurveda_indian_spirituality,This is a pre-1923 historical reproduction tha...,9780090210510,https://covers.openlibrary.org/b/isbn/97800902...
355,Autobiography of a Yogi,"P. Yogananda, W. Y. Evans-Wentz",ayurveda_indian_spirituality,Regarded as one of the most spiritual books of...,0464853028,https://covers.openlibrary.org/b/isbn/04648530...
356,Autobiography of a YOGI,ParamhansaParamhansa Yogananda,ayurveda_indian_spirituality,Yogananda wrote down his Aims and Ideals for S...,1519596677,https://covers.openlibrary.org/b/isbn/15195966...
357,Autobiography of a Yogi,Paramhansa Paramhansa Yogananda,ayurveda_indian_spirituality,The characteristic features of Indian culture ...,9781537524016,https://covers.openlibrary.org/b/isbn/97815375...



WORK: upanishads
RECORDS: 12


,title,author,collection_theme,description_final,isbn_clean,cover_url
184,The Upanishads,Swami Paramananda,yoga_philosophy,"This is a translation of Isha, Katha and Kena ...",9781521509142,https://covers.openlibrary.org/b/isbn/97815215...
185,Upanishads,Acharya Prashant,yoga_philosophy,"Upanishads, in the words of Adi Shankaracharya...",1636333982,https://covers.openlibrary.org/b/isbn/16363339...
187,The Upanishads,Juan Mascaró,yoga_philosophy,NaN,NaN,NaN
191,The Upanishads,"Juan Mascaró, Anonymous",yoga_philosophy,"The Upanishads, the earliest of which were com...",9780140441635,https://covers.openlibrary.org/b/isbn/97801404...
194,The Upanishads,Anonymous,yoga_philosophy,An “Upanisad” is a teaching session with a gur...,9780140447491,https://covers.openlibrary.org/b/isbn/97801404...
196,The Upanishads,Swami Prabhavananda,yoga_philosophy,NaN,0451609212,https://covers.openlibrary.org/b/isbn/04516092...
197,The Upanishads,NaN,yoga_philosophy,The Upanishads: one of three new editions of t...,NaN,NaN
200,The Upanishads,Aurobindo Ghose,yoga_philosophy,The Upanishads are the concluding part of the ...,9788170580034,https://covers.openlibrary.org/b/isbn/97881705...
201,Upanishads,"Swami Prabhavananda, Frederick Manchester",yoga_philosophy,A collection of ancient Hindu mystical writing...,9780451528483,https://covers.openlibrary.org/b/isbn/97804515...
202,The Upanishads,"Jacob Needleman, Eknath Easwaran",yoga_philosophy,"Among the oldest of India's spiritual texts, t...",9781586380250,https://covers.openlibrary.org/b/isbn/97815863...



WORK: yoga sutras of patanjali
RECORDS: 8


...

,title,author,collection_theme,description_final,isbn_clean,cover_url
109,Deep ecology,Bill Devall,humans_and_nature,"Explores the philosophical, psychological, and...",0879051582,https://covers.openlibrary.org/b/isbn/08790515...
113,Deep Ecology,"Bill Devall, George Sessions",humans_and_nature,"Deep Ecology explores the philosophical, psych...",0879052473,https://covers.openlibrary.org/b/isbn/08790524...



WORK: indigenous environmental knowledge and its transformations
RECORDS: 2


,title,author,collection_theme,description_final,isbn_clean,cover_url
439,Indigenous Environmental Knowledge and its Tra...,Roy Ellen,indigenous_knowledge_americas,The first concerted critical examination of th...,9789057024849,https://covers.openlibrary.org/b/isbn/97890570...
441,Indigenous environmental knowledge and its tra...,"R. F. Ellen, Alan Bicker",indigenous_knowledge_americas,First Published in 2000. Routledge is an impri...,9789057024832,https://covers.openlibrary.org/b/isbn/97890570...



WORK: indian spirituality
RECORDS: 2


,title,author,collection_theme,description_final,isbn_clean,cover_url
339,Indian spirituality,Augustine Thottakara,ayurveda_indian_spirituality,NaN,9788186861790,https://covers.openlibrary.org/b/isbn/97881868...
342,Indian Spirituality,Joshua R. Paszkiewicz,ayurveda_indian_spirituality,Nurture your well-being with the eye-opening b...,1577154258,https://covers.openlibrary.org/b/isbn/15771542...


## 7. Strong duplicate edition candidates

This section flags records that have the same normalized work title and a highly similar normalized author.

A high score is a review signal, not an automatic deletion rule.

In [ ]:
duplicate_pairs = []

for work_key, group in duplicate_audit.groupby("work_key"):

    if len(group) < 2:
        continue

    rows = list(group.iterrows())

    for i in range(len(rows)):
        idx_a, row_a = rows[i]

        for j in range(i + 1, len(rows)):
            idx_b, row_b = rows[j]

            author_similarity = SequenceMatcher(
                None,
                row_a["author_key"],
                row_b["author_key"]
            ).ratio()

            duplicate_pairs.append({
                "work_key": work_key,
                "index_a": idx_a,
                "title_a": row_a["title"],
                "author_a": row_a["author"],
                "index_b": idx_b,
                "title_b": row_b["title"],
                "author_b": row_b["author"],
                "author_similarity": round(author_similarity, 3),
            })

duplicate_pairs = pd.DataFrame(duplicate_pairs)

strong_duplicate_pairs = duplicate_pairs[
    duplicate_pairs["author_similarity"] >= 0.75
].sort_values(
    ["work_key", "author_similarity"],
    ascending=[True, False]
)

print("Strong duplicate edition pairs:", len(strong_duplicate_pairs))
display(strong_duplicate_pairs.head(100))

Strong duplicate edition pairs: 43


,work_key,index_a,title_a,author_a,index_b,title_b,author_b,author_similarity
22,autobiography of yogi,346,An Autobiography of a Yogi,Paramhansa Yogananda,360,autobiography of a yogi,paramhansa yogananda,1.000
48,autobiography of yogi,350,Autobiography of a YOGI,Paramahansa Yogananda,359,autobiography of a yogi,paramahansa yogananda,1.000
68,autobiography of yogi,356,Autobiography of a YOGI,ParamhansaParamhansa Yogananda,357,Autobiography of a Yogi,Paramhansa Paramhansa Yogananda,0.984
14,autobiography of yogi,346,An Autobiography of a Yogi,Paramhansa Yogananda,350,Autobiography of a YOGI,Paramahansa Yogananda,0.976
21,autobiography of yogi,346,An Autobiography of a Yogi,Paramhansa Yogananda,359,autobiography of a yogi,paramahansa yogananda,0.976
33,autobiography of yogi,349,Autobiography of a Yogi,Pramahansa Yogananda,350,Autobiography of a YOGI,Paramahansa Yogananda,0.976
40,autobiography of yogi,349,Autobiography of a Yogi,Pramahansa Yogananda,359,autobiography of a yogi,paramahansa yogananda,0.976
43,autobiography of yogi,350,Autobiography of a YOGI,Paramahansa Yogananda,354,Autobiography of a Yogi,PARAMAHANSA YOGANADA,0.976
49,autobiography of yogi,350,Autobiography of a YOGI,Paramahansa Yogananda,360,autobiography of a yogi,paramhansa yogananda,0.976
61,autobiography of yogi,354,Autobiography of a Yogi,PARAMAHANSA YOGANADA,359,autobiography of a yogi,paramahansa yogananda,0.976


## 8. Manual preservation rules

This is where you protect meaningful translations or commentaries from automatic consolidation.

Add work keys for classical or multi edition texts that you want to review manually.

The defaults protect the Bhagavad Gita and Yoga Sutras.

You can add more works later.

In [ ]:
MANUAL_REVIEW_WORKS = {
    "bhagavad gita",
    "yoga sutras of patanjali",
    "yoga sutras patanjali",
}

print("Protected manual review works:")
for work in sorted(MANUAL_REVIEW_WORKS):
    print("•", work)

Protected manual review works:
• bhagavad gita
• yoga sutras of patanjali
• yoga sutras patanjali


## 9. Build a safe deduplication proposal

This creates a **proposal only**.

For repeated works that are not protected for manual review, the notebook selects one representative record using metadata quality.

Preference is given to records with:

1. A useful description
2. A valid ISBN
3. A cover
4. An identified author
5. More complete semantic text

Nothing is removed from the production app at this stage.

In [ ]:
def metadata_quality_score(row):
    score = 0

    description = row.get("description_final")
    if pd.notna(description) and len(str(description).strip()) >= 80:
        score += 4

    isbn = row.get("isbn_clean")
    if pd.notna(isbn) and str(isbn).strip():
        score += 2

    cover = row.get("cover_url")
    if pd.notna(cover) and str(cover).strip():
        score += 2

    author = row.get("author")
    if pd.notna(author) and str(author).strip():
        score += 1

    semantic_words = row.get("semantic_text_words")
    if pd.notna(semantic_words):
        score += min(float(semantic_words) / 100, 2)

    return score

dedup_review = duplicate_audit.copy()
dedup_review["metadata_quality_score"] = dedup_review.apply(
    metadata_quality_score,
    axis=1
)

proposal_rows = []

for work_key, group in dedup_review.groupby("work_key"):

    if len(group) == 1:
        continue

    if work_key in MANUAL_REVIEW_WORKS:
        action = "manual_review_keep_meaningful_editions"
        selected_index = None
    else:
        selected_index = (
            group
            .sort_values("metadata_quality_score", ascending=False)
            .index[0]
        )
        action = "propose_keep_one_representative"

    for idx, row in group.iterrows():
        proposal_rows.append({
            "work_key": work_key,
            "row_index": idx,
            "title": row["title"],
            "author": row["author"],
            "metadata_quality_score": round(row["metadata_quality_score"], 2),
            "proposal_action": action,
            "proposed_keep": (
                True if selected_index == idx
                else False if selected_index is not None
                else None
            ),
        })

dedup_proposal = pd.DataFrame(proposal_rows)

display(
    dedup_proposal.sort_values(
        ["work_key", "metadata_quality_score"],
        ascending=[True, False]
    ).head(150)
)

DEDUP_PROPOSAL_FILE = PROCESSED_DATA / "duplicate_review_proposal.csv"
dedup_proposal.to_csv(DEDUP_PROPOSAL_FILE, index=False)

print("Proposal saved:", DEDUP_PROPOSAL_FILE)

,work_key,row_index,title,author,metadata_quality_score,proposal_action,proposed_keep
1,autobiography of yogi,346,An Autobiography of a Yogi,Paramhansa Yogananda,9,propose_keep_one_representative,True
2,autobiography of yogi,348,Autobiography of a Yogi,Yogananda Paramahamsa,9,propose_keep_one_representative,False
3,autobiography of yogi,349,Autobiography of a Yogi,Pramahansa Yogananda,9,propose_keep_one_representative,False
4,autobiography of yogi,350,Autobiography of a YOGI,Paramahansa Yogananda,9,propose_keep_one_representative,False
5,autobiography of yogi,353,Autobiography of a Yogi,"Paramahansa Paramahansa Yogananda, Adriano Luc...",9,propose_keep_one_representative,False
...,...,...,...,...,...,...,...
131,yoga sutras of patanjali,160,The Yoga Sutras of Patanjali,Sri S. Satchidananda,9,manual_review_keep_meaningful_editions,None
132,yoga sutras of patanjali,162,Yoga Sutras of Patanjali,Swami Vivekananda,9,manual_review_keep_meaningful_editions,None
133,yoga sutras of patanjali,163,The Yoga Sutras Of Patanjali,Alistair Shearer,9,manual_review_keep_meaningful_editions,None
134,yoga sutras of patanjali,165,Yoga sutras of Patanjali,William Q. Judge,9,manual_review_keep_meaningful_editions,None


Proposal saved: /content/drive/MyDrive/kinship_library/data/processed/duplicate_review_proposal.csv


## 10. Estimate the meaningful unique library size

This estimate helps us choose a realistic expansion target.

It counts protected classical works as separate records for now and consolidates only the non protected repeated work groups in the proposal.

This is still an estimate. We will not overwrite the app until the review is complete.

In [ ]:
protected_mask = duplicate_audit["work_key"].isin(MANUAL_REVIEW_WORKS)

non_protected = duplicate_audit[~protected_mask].copy()
protected = duplicate_audit[protected_mask].copy()

estimated_unique_non_protected = (
    non_protected
    .drop_duplicates(subset="work_key")
)

estimated_meaningful_size = (
    len(estimated_unique_non_protected)
    + len(protected)
)

print("Current app rows:", len(app_books))
print("Estimated meaningful books before manual review:", estimated_meaningful_size)
print("Potential redundant rows:", len(app_books) - estimated_meaningful_size)

Current app rows: 713
Estimated meaningful books before manual review: 631
Potential redundant rows: 82


## 11. Curated list guidance after duplicate audit

Now return to **Section 3** and refine your curated list.

Use the curated list mainly for:

**Missing foundational works**  
Example: `Staying with the Trouble`

**Specific translations or commentaries you intentionally want**  
Example: a particular translation of the Bhagavad Gita

**New books you personally want represented**

Do not add a generic foundational title merely because it is important if the catalogue already contains good versions of it.

## 12. Curated anchor audit against the existing catalogue

Before calling Google Books, compare every curated anchor with the current catalogue.

This prevents the curated collection from recreating duplicates. The audit reports the best existing match and assigns one of four review statuses: `ADD`, `REVIEW EXISTING`, `REVIEW SPECIFIC EDITION`, or `REVIEW POSSIBLE MATCH`.

Nothing is deleted or appended here.


In [ ]:
from difflib import SequenceMatcher

def anchor_title_key(value):
    return broad_work_key(value)

def similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

catalogue_for_anchor_audit = app_books.copy()
catalogue_for_anchor_audit["anchor_title_key"] = catalogue_for_anchor_audit["title"].apply(anchor_title_key)
catalogue_for_anchor_audit["anchor_author_key"] = catalogue_for_anchor_audit["author"].apply(normalize_author)

anchor_audit_rows = []

for anchor_id, anchor in curated.reset_index(drop=True).iterrows():
    anchor_key = anchor_title_key(anchor["title"])
    anchor_author_key = normalize_author(anchor["author"])

    scored = catalogue_for_anchor_audit.copy()
    scored["title_similarity"] = scored["anchor_title_key"].apply(lambda x: similarity(anchor_key, x))
    scored["author_similarity"] = (
        scored["anchor_author_key"].apply(lambda x: similarity(anchor_author_key, x))
        if anchor_author_key else 0.0
    )
    scored["match_score"] = 0.85 * scored["title_similarity"] + 0.15 * scored["author_similarity"]
    best = scored.sort_values("match_score", ascending=False).iloc[0]

    title_score = float(best["title_similarity"])
    author_score = float(best["author_similarity"])

    if title_score >= 0.93:
        decision = "REVIEW SPECIFIC EDITION" if anchor_author_key and author_score < 0.60 else "REVIEW EXISTING"
    elif title_score >= 0.82:
        decision = "REVIEW POSSIBLE MATCH"
    else:
        decision = "ADD"

    anchor_audit_rows.append({
        "anchor_id": anchor_id,
        "anchor_title": anchor["title"],
        "anchor_author": anchor["author"],
        "collection_theme": anchor["collection_theme"],
        "best_existing_title": best["title"],
        "best_existing_author": best["author"],
        "title_similarity": round(title_score, 3),
        "author_similarity": round(author_score, 3),
        "match_score": round(float(best["match_score"]), 3),
        "decision": decision,
    })

anchor_audit = pd.DataFrame(anchor_audit_rows)
display(anchor_audit.sort_values(["decision", "match_score"], ascending=[True, False]))
print(anchor_audit["decision"].value_counts())


,anchor_id,anchor_title,anchor_author,collection_theme,best_existing_title,best_existing_author,title_similarity,author_similarity,match_score,decision
9,9,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,Yoga Sastra,Patañjali,0.750,0.385,0.695,ADD
11,11,Vedas & Upanishads,,yoga_philosophy,Upanishads,Eknath Easwaran,0.769,0.000,0.654,ADD
5,5,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,Yoga-Sutra,Kalashatra Govinda,0.690,0.370,0.642,ADD
12,12,Yoga for Americans,Indra Devi,yoga_way_of_life,Yoga for Sex,Vimla Lalvani,0.667,0.435,0.632,ADD
13,13,Yoga Anticolonial Philosophy,Shyam Ranganathan,yoga_philosophy,Yoga Philosophy,"Swami Vivekananda, Patañjali",0.698,0.263,0.632,ADD
17,17,Sri Isopanishad,Bhaktivedanta Swami Prabhupada,yoga_philosophy,Upanishads,Acharya Prashant,0.640,0.450,0.612,ADD
0,0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,Standing Ground,Thomas Buckley,0.667,0.276,0.608,ADD
18,18,Nature's Echo,Thomas Crowther,humans_and_nature,"Man, nature, and ecology",Keith Reid,0.629,0.400,0.594,ADD
6,6,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,Pranayama Yoga Asana,Anand Gupta,0.615,0.387,0.581,ADD
21,21,The Tibetan Book of the Dead,Gyurme Dorje,yoga_philosophy,Textbook of Ayurveda,Vasant Lad,0.650,0.182,0.580,ADD


decision
ADD                        24
REVIEW EXISTING             4
REVIEW SPECIFIC EDITION     1
Name: count, dtype: int64


## 13. Separate missing anchors from books requiring review

Only books classified as `ADD` are prepared for automatic enrichment.

Books already represented in the catalogue remain in a review table so that meaningful translations and editions can be preserved intentionally.


In [ ]:
missing_curated = (
    anchor_audit.loc[anchor_audit["decision"] == "ADD", ["anchor_id"]]
    .merge(
        curated.reset_index().rename(columns={"index": "anchor_id"}),
        on="anchor_id",
        how="left",
    )
    .drop(columns="anchor_id")
)

curated_for_manual_review = anchor_audit[anchor_audit["decision"] != "ADD"].copy()

print("Total curated anchors:", len(curated))
print("Missing anchors ready for enrichment:", len(missing_curated))
print("Anchors requiring review:", len(curated_for_manual_review))

print("\nMISSING CURATED BOOKS")
display(missing_curated)

print("\nREVIEW BEFORE ADDING")
display(curated_for_manual_review)


Total curated anchors: 29
Missing anchors ready for enrichment: 24
Anchors requiring review: 5

MISSING CURATED BOOKS


,title,author,collection_theme,curation_note,_title_key,already_in_app
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,Foundational work for multispecies kinship and...,staying with the trouble,False
1,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,Explores yogic practices and their relationshi...,one simple thing a new look at the science of ...,False
2,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,Foundational philosophical and spiritual text.,yoga vedanta sutras,False
3,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,asana pranayama mudra and bandha,False
4,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,hatha yoga pradipika,False
5,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,Foundational philosophical and spiritual text.,who am i nan yar,False
6,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,Foundational text for Yoga philosophy.,yoga vasistha,False
7,Vedas & Upanishads,,yoga_philosophy,Foundational philosophical and spiritual texts.,vedas upanishads,False
8,Yoga for Americans,Indra Devi,yoga_way_of_life,Historical presentation of Yoga for a broad mo...,yoga for americans,False
9,Yoga Anticolonial Philosophy,Shyam Ranganathan,yoga_philosophy,Anticolonial philosophical approach to Yoga.,yoga anticolonial philosophy,False



REVIEW BEFORE ADDING


,anchor_id,anchor_title,anchor_author,collection_theme,best_existing_title,best_existing_author,title_similarity,author_similarity,match_score,decision
1,1,Autobiography of a Yogi,Paramahansa Yogananda,yoga_philosophy,autobiography of a yogi,paramahansa yogananda,1.0,1.0,1.00,REVIEW EXISTING
2,2,The Yoga Sutras of Patanjali,Patanjali,yoga_philosophy,The Yoga Sutras of Patanjali,Patanjali,1.0,1.0,1.00,REVIEW EXISTING
3,3,Bhagavad Gita,,yoga_philosophy,Bhagavad Gita,Anonymous,1.0,0.0,0.85,REVIEW EXISTING
10,10,The Yoga Sutras of Patanjali,Shyam Ranganathan,yoga_philosophy,The yoga sutras of Patanjali,Satchidananda Swami.,1.0,0.4,0.91,REVIEW SPECIFIC EDITION
19,19,Braiding Sweetgrass,Robin Wall Kimmerer,indigenous_knowledge_americas,Braiding Sweetgrass,Robin Wall Kimmerer,1.0,1.0,1.00,REVIEW EXISTING


## 14. Expansion target after deduplication

The current app has 713 model eligible books. Reaching more than 1000 books means adding at least 288 model eligible books.

I recommend doing this in two layers:

1. Curated foundational books, manually selected in Section 3.
2. Broader API expansion using additional thematic queries.

Do not add hundreds of arbitrary books only to reach a number. The expanded library should remain conceptually coherent.

In [ ]:
# Expansion target

TARGET_LIBRARY_SIZE = 1050

starting_size = globals().get(
    "estimated_meaningful_size",
    len(app_books)
)

books_needed = max(
    0,
    TARGET_LIBRARY_SIZE - starting_size
)

print("Current app rows:", len(app_books))
print("Estimated meaningful starting size:", starting_size)
print("Target:", TARGET_LIBRARY_SIZE)
print("Additional model eligible books needed:", books_needed)

Current app rows: 713
Estimated meaningful starting size: 631
Target: 1050
Additional model eligible books needed: 419


## 15. Expansion queries

You can also edit this cell.

These are broader thematic searches for expanding the catalogue beyond the curated foundational list. They are **search routes**, not definitive classifications.

Add or remove queries depending on the direction you want for Kinship Library.

In [ ]:
EXPANSION_QUERIES = {
    "humans_and_nature": [
        "multispecies kinship",
        "more than human ecology",
        "deep ecology",
        "ecopsychology",
        "human nature relationship",
        "environmental humanities",
        "plant intelligence",
        "animal consciousness",
    ],
    "indigenous_knowledge_americas": [
        "traditional ecological knowledge Americas",
        "Indigenous ecology South America",
        "Andean cosmology",
        "Kogi knowledge",
        "Muisca cosmology",
        "Amazonian Indigenous knowledge",
        "decolonial ecology",
        "Indigenous land relations",
    ],
    "yoga_philosophy": [
        "Yoga Sutras philosophy",
        "Bhagavad Gita philosophy",
        "Vedanta philosophy",
        "Upanishads philosophy",
        "Samkhya philosophy",
        "Indian philosophy consciousness",
    ],
    "ayurveda_indian_spirituality": [
        "Ayurveda philosophy",
        "Indian spirituality",
        "Tantra philosophy",
        "prana subtle body",
        "Hindu ecology",
    ],
    "yoga_way_of_life": [
        "Yoga practice philosophy",
        "meditation practice India",
        "pranayama practice",
        "Yoga ethics daily life",
        "embodied Yoga practice",
    ],
}

for theme, queries in EXPANSION_QUERIES.items():
    print(theme, ":", len(queries), "queries")

humans_and_nature : 8 queries
indigenous_knowledge_americas : 8 queries
yoga_philosophy : 6 queries
ayurveda_indian_spirituality : 5 queries
yoga_way_of_life : 5 queries


## 16. Google Books API setup

This section uses the same type of enrichment workflow as the earlier project.

Store your Google Books API key in Colab Secrets as `GOOGLE_BOOKS_API_KEY`.

This notebook intentionally does not hardcode the key.

In [ ]:
from google.colab import userdata

GOOGLE_BOOKS_API_KEY = userdata.get("API_key_books")

if GOOGLE_BOOKS_API_KEY:
    print("Google Books API key loaded.")
else:
    print("No API key found. Add GOOGLE_BOOKS_API_KEY to Colab Secrets before enrichment.")

Google Books API key loaded.


## 17. Helper functions for Google Books

In [ ]:
import requests

GOOGLE_BOOKS_URL = "https://www.googleapis.com/books/v1/volumes"

def title_similarity(a, b):
    return SequenceMatcher(
        None,
        normalize_title(a),
        normalize_title(b)
    ).ratio()

def extract_isbn(volume_info):
    identifiers = volume_info.get("industryIdentifiers", [])
    isbn13 = None
    isbn10 = None

    for item in identifiers:
        if item.get("type") == "ISBN_13":
            isbn13 = item.get("identifier")
        elif item.get("type") == "ISBN_10":
            isbn10 = item.get("identifier")

    return isbn13 or isbn10

def search_google_book(title, author="", api_key=None):
    query = f'intitle:"{title}"'
    if author:
        query += f' inauthor:"{author}"'

    params = {
        "q": query,
        "maxResults": 5,
        "printType": "books",
    }

    if api_key:
        params["key"] = api_key

    response = requests.get(
        GOOGLE_BOOKS_URL,
        params=params,
        timeout=20
    )
    response.raise_for_status()

    items = response.json().get("items", [])

    if not items:
        return None

    candidates = []

    for item in items:
        info = item.get("volumeInfo", {})
        google_title = info.get("title", "")
        score = title_similarity(title, google_title)

        candidates.append(
            {
                "score": score,
                "google_books_id": item.get("id"),
                "google_title": google_title,
                "google_authors": ", ".join(info.get("authors", [])),
                "description_final": info.get("description"),
                "google_categories": ", ".join(info.get("categories", [])),
                "published_year": str(info.get("publishedDate", ""))[:4],
                "average_rating": info.get("averageRating"),
                "ratings_count": info.get("ratingsCount"),
                "isbn_clean": extract_isbn(info),
                "cover_url": (
                    info.get("imageLinks", {}).get("thumbnail")
                    or info.get("imageLinks", {}).get("smallThumbnail")
                ),
            }
        )

    return max(candidates, key=lambda x: x["score"])

## 18. Enrich missing curated books

Run this only after reviewing Section 3.

The cell searches only for curated books that are not already in the app.

In [ ]:
missing_curated = curated[
    ~curated["already_in_app"]
].copy()

print("Curated books to enrich:", len(missing_curated))

curated_results = []

for _, row in missing_curated.iterrows():
    print("Searching:", row["title"])

    try:
        result = search_google_book(
            title=row["title"],
            author=row["author"],
            api_key=GOOGLE_BOOKS_API_KEY,
        )

        if result is None:
            curated_results.append({
                **row.to_dict(),
                "status": "not_found",
            })
        else:
            curated_results.append({
                **row.to_dict(),
                **result,
                "status": "found",
            })

    except Exception as exc:
        curated_results.append({
            **row.to_dict(),
            "status": "error",
            "error": str(exc),
        })

    time.sleep(0.25)

curated_enriched = pd.DataFrame(curated_results)

display(curated_enriched)
curated_enriched.to_csv(CURATED_ENRICHED_FILE, index=False)

print("Saved:", CURATED_ENRICHED_FILE)

Curated books to enrich: 24
Searching: Staying with the Trouble
Searching: One Simple Thing: A New Look at the Science of Yoga and How It Can Transform Your Life
Searching: Yoga Vedanta Sutras
Searching: Asana, Pranayama, Mudra and Bandha
Searching: Hatha Yoga Pradipika
Searching: Who Am I? (Nan Yar?)
Searching: Yoga Vasistha
Searching: Vedas & Upanishads
Searching: Yoga for Americans
Searching: Yoga Anticolonial Philosophy
Searching: Breath
Searching: Shiva Purana
Searching: Steps to Yoga and Yoga Initiation Papers
Searching: Sri Isopanishad
Searching: Nature's Echo
Searching: The Wisdom of No Escape and the Path of Loving-Kindness
Searching: The Tibetan Book of the Dead
Searching: Eastern Body, Western Mind
Searching: Ancestral Future
Searching: Ideas to Postpone the End of the World
Searching: Life Is Not Useful
Searching: Ch'ixinakax utxiwa
Searching: Un mundo ch'ixi es posible
Searching: Sociología de la imagen


,title,author,collection_theme,curation_note,_title_key,already_in_app,score,google_books_id,google_title,google_authors,description_final,google_categories,published_year,average_rating,ratings_count,isbn_clean,cover_url,status
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,Foundational work for multispecies kinship and...,staying with the trouble,False,1.000000,ZvDgDAAAQBAJ,Staying with the Trouble,Donna J. Haraway,In the midst of spiraling ecological devastati...,Social Science,2016,NaN,NaN,9780822373780,http://books.google.com/books/content?id=ZvDgD...,found
1,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,Explores yogic practices and their relationshi...,one simple thing a new look at the science of ...,False,0.316832,IKZbDwAAQBAJ,One Simple Thing,"Eddie Stern, Deepak Chopra","Yoga was created as a science for liberation, ...",Health & Fitness,2019,NaN,NaN,9780865478404,http://books.google.com/books/content?id=IKZbD...,found
2,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,Foundational philosophical and spiritual text.,yoga vedanta sutras,False,1.000000,Vdz4AAAAIAAJ,Yoga-Vedanta Sutras,Swami Sivananda,None,Vedanta,1955,NaN,NaN,None,http://books.google.com/books/content?id=Vdz4A...,found
3,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,asana pranayama mudra and bandha,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not_found
4,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,Embodied practical Yoga.,hatha yoga pradipika,False,0.512821,uRE-PQAACAAJ,Light on Hatha Yoga,"Swami Muktibodhananda Saraswati, Swami Satyana...",None,Hatha yoga,2000,NaN,NaN,None,None,found
5,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,Foundational philosophical and spiritual text.,who am i nan yar,False,0.666667,tGbxPgAACAAJ,Who Am I.,,None,,2008,NaN,NaN,9788188018048,http://books.google.com/books/content?id=tGbxP...,found
6,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,Foundational text for Yoga philosophy.,yoga vasistha,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not_found
7,Vedas & Upanishads,,yoga_philosophy,Foundational philosophical and spiritual texts.,vedas upanishads,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not_found
8,Yoga for Americans,Indra Devi,yoga_way_of_life,Historical presentation of Yoga for a broad mo...,yoga for americans,False,1.000000,oqUcswEACAAJ,Yoga for Americans,Indra Devi,2015 Reprint of 1959 Edition. Full facsimile o...,,2015,NaN,NaN,9781614278504,http://books.google.com/books/content?id=oqUcs...,found
9,Yoga Anticolonial Philosophy,Shyam Ranganathan,yoga_philosophy,Anticolonial philosophical approach to Yoga.,yoga anticolonial philosophy,False,1.000000,nmnHEAAAQBAJ,Yoga – Anticolonial Philosophy,Shyam Ranganathan,A decolonial guide to yoga from an expert in I...,Health & Fitness,2024,NaN,NaN,9781839978777,http://books.google.com/books/content?id=nmnHE...,found


Saved: /content/drive/MyDrive/kinship_library/data/processed/curated_books_enriched.csv


## 19. Review matches before adding them

Do not automatically accept weak title matches.

A score near 1 means the Google Books title closely matches the requested title.

In [ ]:
# Review Google Books enrichment without rejecting curated books

review = curated_enriched.copy()

def classify_api_match(row):
    status = row.get("status")
    score = row.get("score")

    if status != "found":
        return "CURATED_NO_API_MATCH"

    if pd.isna(score):
        return "CURATED_REVIEW_MATCH"

    if score >= 0.75:
        return "CURATED_GOOD_MATCH"

    return "CURATED_REVIEW_MATCH"


review["curation_status"] = review.apply(
    classify_api_match,
    axis=1
)

review_columns = [
    "title",
    "author",
    "google_title",
    "google_authors",
    "score",
    "status",
    "curation_status",
    "isbn",
]

review_columns = [
    col for col in review_columns
    if col in review.columns
]

display(
    review[review_columns]
    .sort_values(
        ["curation_status", "score"],
        ascending=[True, False]
    )
)

print("\nCURATION STATUS")
print(review["curation_status"].value_counts())

print("\nTotal curated books:", len(review))

,title,author,google_title,google_authors,score,status,curation_status
0,Staying with the Trouble,Donna J. Haraway,Staying with the Trouble,Donna J. Haraway,1.000000,found,CURATED_GOOD_MATCH
2,Yoga Vedanta Sutras,Swami Sivananda,Yoga-Vedanta Sutras,Swami Sivananda,1.000000,found,CURATED_GOOD_MATCH
8,Yoga for Americans,Indra Devi,Yoga for Americans,Indra Devi,1.000000,found,CURATED_GOOD_MATCH
9,Yoga Anticolonial Philosophy,Shyam Ranganathan,Yoga – Anticolonial Philosophy,Shyam Ranganathan,1.000000,found,CURATED_GOOD_MATCH
10,Breath,James Nestor,Breath,James Nestor,1.000000,found,CURATED_GOOD_MATCH
11,Shiva Purana,Bibek Debroy,Shiva Purana,Bibek Debroy,1.000000,found,CURATED_GOOD_MATCH
14,Nature's Echo,Thomas Crowther,Nature's Echo,Thomas Crowther,1.000000,found,CURATED_GOOD_MATCH
18,Ancestral Future,Ailton Krenak,Ancestral Future,Ailton Krenak,1.000000,found,CURATED_GOOD_MATCH
19,Ideas to Postpone the End of the World,Ailton Krenak,Ideas to Postpone the End of the World,Ailton Krenak,1.000000,found,CURATED_GOOD_MATCH
20,Life Is Not Useful,Ailton Krenak,Life Is Not Useful,Ailton Krenak,1.000000,found,CURATED_GOOD_MATCH



CURATION STATUS
curation_status
CURATED_GOOD_MATCH      14
CURATED_NO_API_MATCH     6
CURATED_REVIEW_MATCH     4
Name: count, dtype: int64

Total curated books: 24


## 20. Prepare curated books for the semantic model

Only books with a confident Google Books match and sufficient semantic text should be added automatically.

This keeps the same principle used in the original project.

In [ ]:
# Preserve curated books and use the existing enriched descriptions

curated_library = review.copy()


# --------------------------------------------------
# 1. Preserve curated identity
# --------------------------------------------------

curated_library["title_final"] = (
    curated_library["title"]
)

curated_library["author_final"] = (
    curated_library["author"]
)


# --------------------------------------------------
# 2. Preserve the existing description_final
# --------------------------------------------------

curated_library["description_final"] = (
    curated_library["description_final"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# --------------------------------------------------
# 3. Build semantic text
# --------------------------------------------------

curated_library["semantic_text"] = (
    curated_library["title_final"]
    .fillna("")
    .astype(str)
    + ". "
    + curated_library["author_final"]
    .fillna("")
    .astype(str)
    + ". "
    + curated_library["description_final"]
    .fillna("")
    .astype(str)
)


# --------------------------------------------------
# 4. Count words
# --------------------------------------------------

curated_library["semantic_text_words"] = (
    curated_library["semantic_text"]
    .str.split()
    .str.len()
)


# --------------------------------------------------
# 5. Determine model eligibility
# --------------------------------------------------

MIN_SEMANTIC_WORDS = 15

curated_library["model_eligible"] = (
    curated_library["semantic_text_words"]
    >= MIN_SEMANTIC_WORDS
)


# --------------------------------------------------
# 6. Separate model ready books
# --------------------------------------------------

curated_ready = curated_library[
    curated_library["model_eligible"]
].copy()

curated_needs_metadata = curated_library[
    ~curated_library["model_eligible"]
].copy()


# --------------------------------------------------
# 7. Results
# --------------------------------------------------

print("CURATED LIBRARY")
print(
    "Total enriched curated books:",
    len(curated_library)
)

print(
    "Ready for semantic embeddings:",
    len(curated_ready)
)

print(
    "Need additional descriptive metadata:",
    len(curated_needs_metadata)
)


print("\nREADY FOR EMBEDDINGS")

display(
    curated_ready[
        [
            "title_final",
            "author_final",
            "collection_theme",
            "curation_status",
            "semantic_text_words"
        ]
    ]
)


print("\nCURATED BOOKS NEEDING MORE METADATA")

display(
    curated_needs_metadata[
        [
            "title_final",
            "author_final",
            "collection_theme",
            "curation_status",
            "semantic_text_words"
        ]
    ]
)

CURATED LIBRARY
Total enriched curated books: 24
Ready for semantic embeddings: 12
Need additional descriptive metadata: 12

READY FOR EMBEDDINGS


,title_final,author_final,collection_theme,curation_status,semantic_text_words
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,CURATED_GOOD_MATCH,163
1,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,CURATED_REVIEW_MATCH,223
8,Yoga for Americans,Indra Devi,yoga_way_of_life,CURATED_GOOD_MATCH,241
9,Yoga Anticolonial Philosophy,Shyam Ranganathan,yoga_philosophy,CURATED_GOOD_MATCH,51
10,Breath,James Nestor,yoga_way_of_life,CURATED_GOOD_MATCH,222
11,Shiva Purana,Bibek Debroy,yoga_way_of_life,CURATED_GOOD_MATCH,99
14,Nature's Echo,Thomas Crowther,humans_and_nature,CURATED_GOOD_MATCH,359
15,The Wisdom of No Escape and the Path of Loving...,Pema Chodron,yoga_way_of_life,CURATED_REVIEW_MATCH,203
18,Ancestral Future,Ailton Krenak,indigenous_knowledge_americas,CURATED_GOOD_MATCH,221
19,Ideas to Postpone the End of the World,Ailton Krenak,indigenous_knowledge_americas,CURATED_GOOD_MATCH,260



CURATED BOOKS NEEDING MORE METADATA


,title_final,author_final,collection_theme,curation_status,semantic_text_words
2,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,CURATED_GOOD_MATCH,5
3,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,CURATED_NO_API_MATCH,8
4,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,CURATED_REVIEW_MATCH,6
5,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,CURATED_REVIEW_MATCH,9
6,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,CURATED_NO_API_MATCH,4
7,Vedas & Upanishads,,yoga_philosophy,CURATED_NO_API_MATCH,4
12,Steps to Yoga and Yoga Initiation Papers,Swami Satyananda Saraswati,yoga_way_of_life,CURATED_NO_API_MATCH,10
13,Sri Isopanishad,Bhaktivedanta Swami Prabhupada,yoga_philosophy,CURATED_GOOD_MATCH,5
16,The Tibetan Book of the Dead,Gyurme Dorje,yoga_philosophy,CURATED_NO_API_MATCH,8
17,"Eastern Body, Western Mind",Anodea Judith,yoga_philosophy,CURATED_NO_API_MATCH,6


In [ ]:
print("Rows in curated_to_enrich:", len(curated_to_enrich))
print("Rows in curated_enriched:", len(curated_enriched))
print("Rows in review:", len(review))

print("\nColumns in curated_enriched:")
for col in curated_enriched.columns:
    print(col)

Rows in curated_to_enrich: 25
Rows in curated_enriched: 24
Rows in review: 24

Columns in curated_enriched:
title
author
collection_theme
curation_note
_title_key
already_in_app
score
google_books_id
google_title
google_authors
description_final
google_categories
published_year
average_rating
ratings_count
isbn_clean
cover_url
status


In [ ]:
# Find which curated book disappeared during enrichment

def simple_key(value):
    return (
        str(value)
        .lower()
        .strip()
    )

expected = curated_to_enrich.copy()
actual = curated_enriched.copy()

expected["_check_key"] = (
    expected["title"].apply(simple_key)
    + " | "
    + expected["author"].fillna("").apply(simple_key)
)

actual["_check_key"] = (
    actual["title"].apply(simple_key)
    + " | "
    + actual["author"].fillna("").apply(simple_key)
)

missing_after_enrichment = expected[
    ~expected["_check_key"].isin(actual["_check_key"])
].copy()

print(
    "Books missing after enrichment:",
    len(missing_after_enrichment)
)

display(
    missing_after_enrichment[
        [
            "title",
            "author",
            "collection_theme",
            "curation_note"
        ]
    ]
)

Books missing after enrichment: 1


,title,author,collection_theme,curation_note
24,The Yoga Sutras of Patanjali,Shyam Ranganathan,yoga_philosophy,Specific philosophical translation and interpr...


In [ ]:
# Restore the intentionally selected Shyam Ranganathan edition

ranganathan_book = missing_after_enrichment.copy()

ranganathan_book["score"] = None
ranganathan_book["google_books_id"] = None
ranganathan_book["google_title"] = None
ranganathan_book["google_authors"] = None
ranganathan_book["description_final"] = ""
ranganathan_book["google_categories"] = None
ranganathan_book["published_year"] = None
ranganathan_book["average_rating"] = None
ranganathan_book["ratings_count"] = None
ranganathan_book["isbn_clean"] = None
ranganathan_book["cover_url"] = None

ranganathan_book["status"] = "not_found"
ranganathan_book["curation_status"] = "CURATED_NO_API_MATCH"

ranganathan_book["title_final"] = ranganathan_book["title"]
ranganathan_book["author_final"] = ranganathan_book["author"]

ranganathan_book["semantic_text"] = (
    ranganathan_book["title_final"].fillna("").astype(str)
    + ". "
    + ranganathan_book["author_final"].fillna("").astype(str)
)

ranganathan_book["semantic_text_words"] = (
    ranganathan_book["semantic_text"]
    .str.split()
    .str.len()
)

ranganathan_book["model_eligible"] = False

curated_library = pd.concat(
    [curated_library, ranganathan_book],
    ignore_index=True
)

print("Total curated books preserved:", len(curated_library))

display(
    curated_library[
        [
            "title_final",
            "author_final",
            "collection_theme",
            "curation_status",
            "model_eligible"
        ]
    ].tail()
)

Total curated books preserved: 25


/tmp/ipykernel_929/1749140701.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  curated_library = pd.concat(


,title_final,author_final,collection_theme,curation_status,model_eligible
20,Life Is Not Useful,Ailton Krenak,indigenous_knowledge_americas,CURATED_GOOD_MATCH,True
21,Ch'ixinakax utxiwa,Silvia Rivera Cusicanqui,indigenous_knowledge_americas,CURATED_GOOD_MATCH,True
22,Un mundo ch'ixi es posible,Silvia Rivera Cusicanqui,indigenous_knowledge_americas,CURATED_GOOD_MATCH,False
23,Sociología de la imagen,Silvia Rivera Cusicanqui,indigenous_knowledge_americas,CURATED_GOOD_MATCH,False
24,The Yoga Sutras of Patanjali,Shyam Ranganathan,yoga_philosophy,CURATED_NO_API_MATCH,False


In [ ]:
print("Total curated books preserved:", len(curated_library))

Total curated books preserved: 25


In [ ]:
curated_needs_metadata = curated_library[
    ~curated_library["model_eligible"]
].copy()

print(
    "Curated books needing descriptive metadata:",
    len(curated_needs_metadata)
)

display(
    curated_needs_metadata[
        [
            "title_final",
            "author_final",
            "collection_theme",
            "curation_status",
            "semantic_text_words"
        ]
    ].reset_index(drop=True)
)

Curated books needing descriptive metadata: 13


,title_final,author_final,collection_theme,curation_status,semantic_text_words
0,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,CURATED_GOOD_MATCH,5
1,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,CURATED_NO_API_MATCH,8
2,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,CURATED_REVIEW_MATCH,6
3,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,CURATED_REVIEW_MATCH,9
4,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,CURATED_NO_API_MATCH,4
5,Vedas & Upanishads,,yoga_philosophy,CURATED_NO_API_MATCH,4
6,Steps to Yoga and Yoga Initiation Papers,Swami Satyananda Saraswati,yoga_way_of_life,CURATED_NO_API_MATCH,10
7,Sri Isopanishad,Bhaktivedanta Swami Prabhupada,yoga_philosophy,CURATED_GOOD_MATCH,5
8,The Tibetan Book of the Dead,Gyurme Dorje,yoga_philosophy,CURATED_NO_API_MATCH,8
9,"Eastern Body, Western Mind",Anodea Judith,yoga_philosophy,CURATED_NO_API_MATCH,6


In [ ]:
#Cleaning variables before embeddings
# Remove outdated expansion variables from the previous 10 book run

for variable in [
    "expanded_books",
    "expanded_embeddings",
    "curated_embeddings"
]:
    if variable in globals():
        del globals()[variable]

print("Old temporary expansion variables cleared.")

Old temporary expansion variables cleared.


## 20.1 Adding the missing metadata from a ChatGPT research to include the curated books

In [ ]:
# ============================================================
# Curated metadata recovery
# ============================================================

CURATED_METADATA_RECOVERY = {

    "Yoga Vedanta Sutras": {
        "description": (
            "A work in the Yoga Vedanta tradition associated with Swami "
            "Sivananda, bringing together teachings on Yoga, Vedanta, "
            "spiritual discipline, self knowledge, meditation and the "
            "realization of the divine nature of the Self."
        ),
        "description_source": "Divine Life Society",
        "description_source_url": "https://www.dlshq.org/books/",
        "metadata_method": "curated_verified_summary",
    },

    "Asana, Pranayama, Mudra and Bandha": {
        "description": (
            "A comprehensive practical manual of the Satyananda Yoga "
            "tradition presenting asana, pranayama, mudra, bandha and "
            "shatkarma. The book moves from foundational to advanced "
            "practices and also introduces yogic physiology, meditation, "
            "relaxation and the practical application of Yoga in daily life."
        ),
        "description_source": "Yoga Publications Trust and Google Books",
        "description_source_url":
            "https://books.google.com/books?id=7z2pzwEACAAJ",
        "metadata_method": "curated_verified_summary",
    },

    "Hatha Yoga Pradipika": {
        "description": (
            "A foundational text of Hatha Yoga concerned with embodied "
            "practice and the transformation of body, breath and mind. "
            "Its teachings include asana, pranayama, mudra, bandha, "
            "meditative absorption and the energetic dimensions of "
            "classical Hatha Yoga practice."
        ),
        "description_source": "Curated classical text summary",
        "description_source_url": "",
        "metadata_method": "curated_research_summary",
    },

    "Who Am I? (Nan Yar?)": {
        "description": (
            "A concise presentation of Ramana Maharshi's teaching of "
            "self inquiry. Originating in questions posed to him in the "
            "early twentieth century, the text investigates the source "
            "of the sense of I and presents self inquiry as a direct "
            "path toward recognition of one's true nature."
        ),
        "description_source": "Advaita Media and Ramana teaching archive",
        "description_source_url":
            "https://advaitamedia.com/en/produkt/nan-yar-who-am-i-2/",
        "metadata_method": "curated_verified_summary",
    },

    "Yoga Vasistha": {
        "description": (
            "A major Sanskrit philosophical work presented as a dialogue "
            "between the sage Vasistha and Rama. The text explores "
            "consciousness, liberation, mind, reality, nonduality and "
            "self knowledge through philosophical discussion, stories "
            "and contemplative teaching. This record refers to the "
            "English translation by Vihari Lala Mitra."
        ),
        "description_source": "Project Gutenberg, Vihari Lala Mitra translation",
        "description_source_url":
            "https://www.gutenberg.org/files/71326/71326-h/71326-h.htm",
        "metadata_method": "curated_verified_summary",
    },

    "Vedas & Upanishads": {
        "description": (
            "A contemporary selection of teachings from the Vedas and "
            "Upanishads addressing dharma, karma, Atman, Brahman, "
            "detachment, awareness, liberation and self inquiry. The "
            "volume presents ancient Indian philosophical teachings for "
            "readers interested in meaning, inner clarity and spiritual "
            "reflection."
        ),
        "description_source": "Penguin Random House",
        "description_source_url":
            "https://www.penguinrandomhouse.com/books/832240/vedas-and-upanishads-greatest-spiritual-wisdom-for-tough-times-by-pranay/",
        "metadata_method": "curated_verified_summary",
    },

    "Steps to Yoga and Yoga Initiation Papers": {
        "description": (
            "A collection of letters and instructions written by Swami "
            "Satyananda Saraswati to disciples during their spiritual "
            "practice. The teachings explore yoga sadhana, awakening, "
            "discipleship and the relationship between guru and student, "
            "bringing philosophical Yoga into conversation with the "
            "practical challenges of spiritual life."
        ),
        "description_source": "Yoga Publications Trust and Google Books",
        "description_source_url":
            "https://books.google.com/books?id=airLPAAACAAJ",
        "metadata_method": "curated_verified_summary",
    },

    "Sri Isopanishad": {
        "description": (
            "A translation and commentary on the Isha Upanishad by "
            "A. C. Bhaktivedanta Swami Prabhupada. Its eighteen mantras "
            "consider the divine as the complete whole of existence and "
            "explore the relationship between the Supreme, living beings "
            "and the world through Vedic philosophy and devotional thought."
        ),
        "description_source": "Bhaktivedanta Book Trust",
        "description_source_url":
            "https://bbt.org/book/sri-isopanisad",
        "metadata_method": "curated_verified_summary",
    },

    "The Tibetan Book of the Dead": {
        "description": (
            "Gyurme Dorje's complete English translation of the Tibetan "
            "Buddhist text traditionally concerned with the processes of "
            "living, dying and the intermediate states after death. It "
            "combines contemplative practice, guidance for the dying and "
            "bereaved, and philosophical reflections on consciousness, "
            "death and liberation."
        ),
        "description_source": "Penguin Classics",
        "description_source_url":
            "https://www.penguinrandomhouse.com/books/288210/the-tibetan-book-of-the-dead-by-gyurme-dorje/",
        "metadata_method": "curated_verified_summary",
    },

    "Eastern Body, Western Mind": {
        "description": (
            "Anodea Judith brings the Eastern chakra system into dialogue "
            "with Western psychology, Jungian thought, somatic therapy "
            "and developmental theory. The chakras become a framework "
            "for exploring psychological development, embodiment, trauma, "
            "relationships, personal growth and the integration of mind, "
            "body and spirituality."
        ),
        "description_source": "Penguin Random House",
        "description_source_url":
            "https://www.penguinrandomhouse.com/books/197177/eastern-body-western-mind-by-anodea-judith/",
        "metadata_method": "curated_verified_summary",
    },

    "Un mundo ch'ixi es posible": {
        "description": (
            "Silvia Rivera Cusicanqui develops ch'ixi as an Andean and "
            "decolonial way of thinking about coexistence without reducing "
            "difference to a single identity. The essays connect Indigenous "
            "thought, epistemology, territory, colonialism, everyday life "
            "and political practice while imagining forms of coexistence "
            "rooted in multiple worlds and ways of knowing."
        ),
        "description_source": "Tinta Limón Ediciones",
        "description_source_url":
            "https://tintalimon.com.ar/libro/un-mundo-ch-ixi-es-posible/",
        "metadata_method": "curated_verified_summary",
    },

    "Sociología de la imagen": {
        "description": (
            "Silvia Rivera Cusicanqui develops a sociology of the image "
            "rooted in Andean history and ch'ixi thought. The work explores "
            "images as forms of memory and knowledge, critiques colonial "
            "ways of seeing, and reconnects visual interpretation with "
            "embodied senses, historical experience, artistic practice "
            "and decolonial political reflection."
        ),
        "description_source": "Tinta Limón Ediciones",
        "description_source_url":
            "https://tintalimon.com.ar/libro/sociolog%C3%ADa-de-la-imagen/",
        "metadata_method": "curated_verified_summary",
    },

    "The Yoga Sutras of Patanjali": {
        "description": (
            "Shyam Ranganathan's translation and commentary approaches "
            "Patanjali's Yoga Sutra as a work of philosophy and ethical "
            "inquiry. His interpretation emphasizes philosophical method, "
            "moral thought and the challenge of translating Sanskrit "
            "philosophy without simply projecting the assumptions of the "
            "translator onto the original text."
        ),
        "description_source": "Yoga Philosophy Institute",
        "description_source_url":
            "https://yogaphilosophy.com/product/patanjalis-yoga-sutra/",
        "metadata_method": "curated_verified_summary",
    },
}

In [ ]:
# ============================================================
# Apply recovered metadata to curated_library
# ============================================================

for title, metadata in CURATED_METADATA_RECOVERY.items():

    mask = (
        curated_library["title_final"]
        .str.strip()
        .str.lower()
        == title.strip().lower()
    )

    if mask.sum() == 0:
        print("Not found in curated_library:", title)
        continue

    curated_library.loc[
        mask,
        "description_final"
    ] = metadata["description"]

    curated_library.loc[
        mask,
        "description_source"
    ] = metadata["description_source"]

    curated_library.loc[
        mask,
        "description_source_url"
    ] = metadata["description_source_url"]

    curated_library.loc[
        mask,
        "metadata_method"
    ] = metadata["metadata_method"]


# Correct the specific Vedas & Upanishads record
vedas_mask = (
    curated_library["title_final"]
    .str.strip()
    .str.lower()
    == "vedas & upanishads"
)

curated_library.loc[
    vedas_mask,
    "author_final"
] = "Pranay"


# Rebuild semantic text
curated_library["semantic_text"] = (
    curated_library["title_final"]
    .fillna("")
    .astype(str)
    + ". "
    + curated_library["author_final"]
    .fillna("")
    .astype(str)
    + ". "
    + curated_library["description_final"]
    .fillna("")
    .astype(str)
)


# Recalculate word count
curated_library["semantic_text_words"] = (
    curated_library["semantic_text"]
    .str.split()
    .str.len()
)


# Recalculate model eligibility
MIN_SEMANTIC_WORDS = 15

curated_library["model_eligible"] = (
    curated_library["semantic_text_words"]
    >= MIN_SEMANTIC_WORDS
)


# Rebuild final model ready set
curated_ready = curated_library[
    curated_library["model_eligible"]
].copy()


# Check remaining problems
curated_needs_metadata = curated_library[
    ~curated_library["model_eligible"]
].copy()


print("Total curated books:", len(curated_library))
print("Ready for semantic embeddings:", len(curated_ready))
print(
    "Still needing metadata:",
    len(curated_needs_metadata)
)

display(
    curated_ready[
        [
            "title_final",
            "author_final",
            "collection_theme",
            "semantic_text_words",
            "metadata_method"
        ]
    ]
)

Total curated books: 25
Ready for semantic embeddings: 25
Still needing metadata: 0


,title_final,author_final,collection_theme,semantic_text_words,metadata_method
0,Staying with the Trouble,Donna J. Haraway,humans_and_nature,163,NaN
1,One Simple Thing: A New Look at the Science of...,Eddie Stern,yoga_way_of_life,223,NaN
2,Yoga Vedanta Sutras,Swami Sivananda,yoga_philosophy,37,curated_verified_summary
3,"Asana, Pranayama, Mudra and Bandha",Swami Satyananda Saraswati,yoga_way_of_life,48,curated_verified_summary
4,Hatha Yoga Pradipika,Swami Satyananda Saraswati,yoga_way_of_life,42,curated_research_summary
5,Who Am I? (Nan Yar?),Bhagavan Sri Ramana Maharshi,yoga_way_of_life,54,curated_verified_summary
6,Yoga Vasistha,Vihari-Lala Mitra,yoga_philosophy,48,curated_verified_summary
7,Vedas & Upanishads,Pranay,yoga_philosophy,42,curated_verified_summary
8,Yoga for Americans,Indra Devi,yoga_way_of_life,241,NaN
9,Yoga Anticolonial Philosophy,Shyam Ranganathan,yoga_philosophy,51,NaN


## 21. Generate embeddings with the same model

This uses exactly the same Sentence Transformer model as the current app.

Do not change the model name unless you intend to regenerate the entire library.

In [ ]:
# ============================================================
# FINAL CURATED EMBEDDINGS
# ============================================================

# Make sure the order is fixed before embedding
curated_ready = (
    curated_library[
        curated_library["model_eligible"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Curated books to embed:", len(curated_ready))


# Generate embeddings using the SAME SentenceTransformer model
curated_embeddings = model.encode(
    curated_ready["semantic_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Curated embeddings shape:", curated_embeddings.shape)


# Alignment check
assert len(curated_ready) == len(curated_embeddings)
assert curated_embeddings.shape[1] == 384

print("Curated books and embeddings are aligned.")

Curated books to embed: 25


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Curated embeddings shape: (25, 384)
Curated books and embeddings are aligned.


In [ ]:
# ============================================================
# BETTER CURATED VS EXISTING CATALOGUE AUDIT
# Compare normalized titles and inspect authors
# ============================================================

comparison_rows = []

for curated_index, row in curated_check.iterrows():

    title_key = row["_title_key"]

    # Find existing catalogue records with same normalized title
    matches = existing_check[
        existing_check["_title_key"] == title_key
    ]

    if len(matches) == 0:

        comparison_rows.append({
            "curated_index": curated_index,
            "curated_title": row["title_final"],
            "curated_author": row["author_final"],
            "existing_title": None,
            "existing_author": None,
            "title_match": False
        })

    else:

        for _, existing_row in matches.iterrows():

            comparison_rows.append({
                "curated_index": curated_index,
                "curated_title": row["title_final"],
                "curated_author": row["author_final"],
                "existing_title":
                    existing_row[existing_title_col],
                "existing_author":
                    existing_row[existing_author_col],
                "title_match": True
            })


title_audit = pd.DataFrame(comparison_rows)


print(
    "Curated books with at least one title match:",
    title_audit[
        title_audit["title_match"]
    ]["curated_index"].nunique()
)

print(
    "Curated books with no title match:",
    25 - title_audit[
        title_audit["title_match"]
    ]["curated_index"].nunique()
)


print("\nPOSSIBLE EXISTING WORKS")

display(
    title_audit[
        title_audit["title_match"]
    ][
        [
            "curated_index",
            "curated_title",
            "curated_author",
            "existing_title",
            "existing_author"
        ]
    ].reset_index(drop=True)
)

Curated books with at least one title match: 1
Curated books with no title match: 24

POSSIBLE EXISTING WORKS


,curated_index,curated_title,curated_author,existing_title,existing_author
0,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,The Yoga Sutras of Patanjali,Patanjali
1,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,The yoga sutras of Patanjali,Satchidananda Swami.
2,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,Yoga Sutras of Patanjali,Charles Johnston
3,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,Yoga Sutras of Patanjali,"Charles Johnston, Christopher Harding"
4,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,The Yoga Sutras of Patanjali,Sri S. Satchidananda
5,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,The Yoga sutras of Patañjali,Edwin F. Bryant
6,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,Yoga Sutras of Patanjali,Swami Vivekananda
7,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,The Yoga Sutras Of Patanjali,Alistair Shearer
8,24,The Yoga Sutras of Patanjali,Shyam Ranganathan,Yoga sutras of Patanjali,William Q. Judge


In [ ]:
# ============================================================
# FINAL CURATED CATALOGUE MERGE
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Final curated books to add
# ------------------------------------------------------------

curated_to_add = curated_ready.copy().reset_index(drop=True)

print("Existing catalogue:", len(app_books))
print("Curated books to add:", len(curated_to_add))
print("Curated embeddings:", curated_embeddings.shape)


# ------------------------------------------------------------
# 2. Safety checks before merging
# ------------------------------------------------------------

assert len(curated_to_add) == len(curated_embeddings)

assert app_embeddings.shape[0] == len(app_books)

assert app_embeddings.shape[1] == curated_embeddings.shape[1]

print("Pre merge alignment checks passed.")


# ------------------------------------------------------------
# 3. Make curated rows compatible with app_books
# ------------------------------------------------------------

# Use final curated title and author
curated_to_add["title"] = curated_to_add["title_final"]
curated_to_add["author"] = curated_to_add["author_final"]


# Add any columns required by app_books that are absent
for col in app_books.columns:
    if col not in curated_to_add.columns:
        curated_to_add[col] = pd.NA


# Keep exactly the same column structure
curated_for_app = curated_to_add[
    app_books.columns
].copy()


# ------------------------------------------------------------
# 4. Merge catalogue
# ------------------------------------------------------------

expanded_books = pd.concat(
    [
        app_books.reset_index(drop=True),
        curated_for_app.reset_index(drop=True)
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 5. Merge embeddings in identical row order
# ------------------------------------------------------------

expanded_embeddings = np.vstack(
    [
        app_embeddings,
        curated_embeddings
    ]
)


# ------------------------------------------------------------
# 6. Final alignment checks
# ------------------------------------------------------------

assert len(expanded_books) == expanded_embeddings.shape[0]

assert expanded_embeddings.shape[1] == 384


print("\nFINAL CURATED EXPANSION")
print("Books before:", len(app_books))
print("Books added:", len(curated_for_app))
print("Books after:", len(expanded_books))
print("Embedding matrix:", expanded_embeddings.shape)
print(
    "Catalogue and embeddings aligned:",
    len(expanded_books) == len(expanded_embeddings)
)

Existing catalogue: 713
Curated books to add: 25
Curated embeddings: (25, 384)
Pre merge alignment checks passed.

FINAL CURATED EXPANSION
Books before: 713
Books added: 25
Books after: 738
Embedding matrix: (738, 384)
Catalogue and embeddings aligned: True


In [ ]:
# ============================================================
# SAVE CURATED EXPANSION CHECKPOINT
# ============================================================

import os
import numpy as np

checkpoint_dir = (
    "/content/drive/MyDrive/kinship_library/"
    "data/processed"
)

model_dir = (
    "/content/drive/MyDrive/kinship_library/"
    "models"
)

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)


# Save catalogue checkpoint
expanded_books.to_csv(
    f"{checkpoint_dir}/books_curated_expanded.csv",
    index=False
)


# Save aligned embeddings checkpoint
np.save(
    f"{model_dir}/book_embeddings_curated_expanded.npy",
    expanded_embeddings
)


print("Checkpoint saved.")
print("Books:", len(expanded_books))
print("Embeddings:", expanded_embeddings.shape)

Checkpoint saved.
Books: 738
Embeddings: (738, 384)


## 22. Library Expansion

In [ ]:
# ============================================================
# BROADER LIBRARY EXPANSION
# Step 1: thematic discovery queries
# ============================================================

EXPANSION_QUERIES = {

    "humans_and_nature": [
        "multispecies kinship",
        "more than human ecology",
        "human nature relationship",
        "environmental humanities",
        "ecological philosophy",
        "ecopsychology",
        "deep ecology",
        "plant intelligence",
        "animal consciousness",
        "forest ecology philosophy",
        "ecological grief",
        "ecological belonging",
        "living systems philosophy",
        "regenerative ecology",
        "nature culture philosophy",
    ],

    "indigenous_knowledge_americas": [
        "Indigenous philosophy Americas",
        "Indigenous philosophy South America",
        "Andean philosophy",
        "Andean epistemology",
        "Andean cosmology",
        "Aymara philosophy",
        "Quechua philosophy",
        "Amazonian Indigenous philosophy",
        "Indigenous relational ontology",
        "Traditional Ecological Knowledge Americas",
        "Indigenous ecology South America",
        "Kogi knowledge",
        "Muisca cosmology",
        "Indigenous land relations",
        "decolonial Indigenous thought",
        "decolonial Andean thought",
        "ch'ixi epistemology",
        "Buen Vivir philosophy",
        "Sumak Kawsay",
        "Abya Yala Indigenous thought",
        "Indigenous environmental philosophy",
        "Indigenous knowledge decolonization",
    ],

    "yoga_philosophy": [
        "Yoga Sutras philosophy",
        "Patanjali Yoga philosophy",
        "Bhagavad Gita philosophy",
        "Vedanta philosophy",
        "Advaita Vedanta",
        "Upanishads philosophy",
        "Samkhya philosophy",
        "Indian philosophy consciousness",
        "Indian philosophy self",
        "Yoga ethics",
        "Yoga liberation philosophy",
        "classical Yoga philosophy",
        "nondual Indian philosophy",
        "Indian philosophy mind",
    ],

    "ayurveda_indian_spirituality": [
        "Ayurveda philosophy",
        "Ayurvedic psychology",
        "Ayurveda mind body",
        "Indian spirituality",
        "Tantra philosophy",
        "Shiva philosophy",
        "Shakti philosophy",
        "Kashmir Shaivism",
        "prana subtle body",
        "Hindu ecology",
        "Indian sacred ecology",
        "Indian philosophy nature",
        "Vedic ecology",
    ],

    "yoga_way_of_life": [
        "Yoga practice philosophy",
        "Yoga way of life",
        "meditation practice India",
        "pranayama",
        "Yoga ethics daily life",
        "embodied Yoga practice",
        "Hatha Yoga practice",
        "Yoga meditation practice",
        "Yoga and daily life",
        "Yoga self inquiry",
        "Yoga spiritual practice",
        "Yoga sadhana",
    ],
}


query_count = sum(
    len(queries)
    for queries in EXPANSION_QUERIES.values()
)

print("Expansion themes:", len(EXPANSION_QUERIES))
print("Total discovery queries:", query_count)

for theme, queries in EXPANSION_QUERIES.items():
    print(
        theme,
        ":",
        len(queries),
        "queries"
    )

Expansion themes: 5
Total discovery queries: 76
humans_and_nature : 15 queries
indigenous_knowledge_americas : 22 queries
yoga_philosophy : 14 queries
ayurveda_indian_spirituality : 13 queries
yoga_way_of_life : 12 queries


In [ ]:
# ============================================================
# Step 2: expansion parameters
# ============================================================

RESULTS_PER_QUERY = 10

TARGET_MEANINGFUL_LIBRARY_SIZE = 1050

estimated_meaningful_current = 631 + 25

estimated_gap = max(
    0,
    TARGET_MEANINGFUL_LIBRARY_SIZE
    - estimated_meaningful_current
)

print(
    "Estimated meaningful books after curation:",
    estimated_meaningful_current
)

print(
    "Target meaningful library size:",
    TARGET_MEANINGFUL_LIBRARY_SIZE
)

print(
    "Estimated meaningful gap:",
    estimated_gap
)

print(
    "Maximum raw API candidates this pass:",
    query_count * RESULTS_PER_QUERY
)

Estimated meaningful books after curation: 656
Target meaningful library size: 1050
Estimated meaningful gap: 394
Maximum raw API candidates this pass: 760


## 23 Creating the new curated collection with API

In [ ]:
# ============================================================
# BROADER LIBRARY EXPANSION
# Step 3: Google Books discovery and raw cache
# ============================================================

import os
import time
import requests
import pandas as pd

from google.colab import userdata


# ------------------------------------------------------------
# 1. API key
# ------------------------------------------------------------

GOOGLE_BOOKS_API_KEY = userdata.get("API_key_books")

if not GOOGLE_BOOKS_API_KEY:
    raise ValueError(
        "GOOGLE_BOOKS_API_KEY was not found in Colab userdata."
    )


# ------------------------------------------------------------
# 2. Cache location
# ------------------------------------------------------------

EXPANSION_CACHE = (
    "/content/drive/MyDrive/kinship_library/"
    "data/raw/google_books_expansion_raw.csv"
)


# ------------------------------------------------------------
# 3. Google Books search function
# ------------------------------------------------------------

def search_google_books_expansion(
    query,
    theme,
    max_results=10
):

    url = "https://www.googleapis.com/books/v1/volumes"

    params = {
        "q": query,
        "maxResults": max_results,
        "printType": "books",
        "key": GOOGLE_BOOKS_API_KEY
    }

    for attempt in range(3):

        try:

            response = requests.get(
                url,
                params=params,
                timeout=20
            )

            if response.status_code == 429:
                print("Quota or rate limit. Waiting...")
                time.sleep(10 * (attempt + 1))
                continue

            response.raise_for_status()

            data = response.json()

            rows = []

            for item in data.get("items", []):

                info = item.get("volumeInfo", {})

                identifiers = info.get(
                    "industryIdentifiers",
                    []
                )

                isbn_13 = None
                isbn_10 = None

                for identifier in identifiers:

                    if identifier.get("type") == "ISBN_13":
                        isbn_13 = identifier.get("identifier")

                    elif identifier.get("type") == "ISBN_10":
                        isbn_10 = identifier.get("identifier")


                image_links = info.get(
                    "imageLinks",
                    {}
                )

                rows.append({

                    "google_books_id":
                        item.get("id"),

                    "title":
                        info.get("title"),

                    "authors":
                        ", ".join(
                            info.get("authors", [])
                        ),

                    "description":
                        info.get("description"),

                    "categories":
                        " | ".join(
                            info.get("categories", [])
                        ),

                    "publisher":
                        info.get("publisher"),

                    "published_date":
                        info.get("publishedDate"),

                    "language":
                        info.get("language"),

                    "average_rating":
                        info.get("averageRating"),

                    "ratings_count":
                        info.get("ratingsCount"),

                    "isbn_13":
                        isbn_13,

                    "isbn_10":
                        isbn_10,

                    "thumbnail":
                        image_links.get("thumbnail"),

                    "collection_theme":
                        theme,

                    "discovery_query":
                        query,

                    "curation_source":
                        "google_books_expansion"
                })

            return rows

        except requests.RequestException as error:

            print(
                f"Attempt {attempt + 1} failed:",
                query,
                error
            )

            time.sleep(3 * (attempt + 1))


    return []


# ------------------------------------------------------------
# 4. Collect every thematic query
# ------------------------------------------------------------

all_expansion_rows = []

total_queries = sum(
    len(queries)
    for queries in EXPANSION_QUERIES.values()
)

query_number = 0


for theme, queries in EXPANSION_QUERIES.items():

    print("\nTHEME:", theme)

    for query in queries:

        query_number += 1

        print(
            f"[{query_number}/{total_queries}]",
            query
        )

        rows = search_google_books_expansion(
            query=query,
            theme=theme,
            max_results=RESULTS_PER_QUERY
        )

        all_expansion_rows.extend(rows)

        print("  Results:", len(rows))

        # Small pause between requests
        time.sleep(0.4)


# ------------------------------------------------------------
# 5. Create raw candidate dataframe
# ------------------------------------------------------------

expansion_raw = pd.DataFrame(
    all_expansion_rows
)


# ------------------------------------------------------------
# 6. Save immediately
# ------------------------------------------------------------

os.makedirs(
    os.path.dirname(EXPANSION_CACHE),
    exist_ok=True
)

expansion_raw.to_csv(
    EXPANSION_CACHE,
    index=False
)


print("\nCOLLECTION COMPLETE")
print("Raw API rows:", len(expansion_raw))
print("Saved to:", EXPANSION_CACHE)


THEME: humans_and_nature
[1/76] multispecies kinship
  Results: 10
[2/76] more than human ecology
  Results: 10
[3/76] human nature relationship
  Results: 10
[4/76] environmental humanities
  Results: 10
[5/76] ecological philosophy
  Results: 10
[6/76] ecopsychology
  Results: 10
[7/76] deep ecology
  Results: 10
[8/76] plant intelligence
  Results: 10
[9/76] animal consciousness
  Results: 10
[10/76] forest ecology philosophy
  Results: 10
[11/76] ecological grief
  Results: 10
[12/76] ecological belonging
  Results: 10
[13/76] living systems philosophy
  Results: 10
[14/76] regenerative ecology
  Results: 10
[15/76] nature culture philosophy
  Results: 10

THEME: indigenous_knowledge_americas
[16/76] Indigenous philosophy Americas
  Results: 10
[17/76] Indigenous philosophy South America
  Results: 10
[18/76] Andean philosophy
  Results: 10
[19/76] Andean epistemology
  Results: 10
[20/76] Andean cosmology
  Results: 10
[21/76] Aymara philosophy
  Results: 10
[22/76] Quechua philo

In [ ]:
# ============================================================
# Step 4: Clean the raw candidate pool
# ============================================================

import re
import unicodedata


def normalize_expansion_text(value):

    if pd.isna(value):
        return ""

    value = str(value).lower().strip()

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )

    value = re.sub(
        r"[^\w\s]",
        " ",
        value
    )

    value = re.sub(
        r"^the\s+",
        "",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


expansion_candidates = expansion_raw.copy()


# ------------------------------------------------------------
# Normalized title and author
# ------------------------------------------------------------

expansion_candidates["_title_key"] = (
    expansion_candidates["title"]
    .apply(normalize_expansion_text)
)

expansion_candidates["_author_key"] = (
    expansion_candidates["authors"]
    .apply(normalize_expansion_text)
)

expansion_candidates["_work_key"] = (
    expansion_candidates["_title_key"]
    + " | "
    + expansion_candidates["_author_key"]
)


# ------------------------------------------------------------
# Remove unusable records
# ------------------------------------------------------------

before = len(expansion_candidates)

expansion_candidates = (
    expansion_candidates[
        expansion_candidates["_title_key"] != ""
    ]
    .copy()
)

print(
    "Removed records without title:",
    before - len(expansion_candidates)
)


# ------------------------------------------------------------
# First deduplicate by Google Books ID
# ------------------------------------------------------------

before = len(expansion_candidates)

expansion_candidates = (
    expansion_candidates
    .drop_duplicates(
        subset="google_books_id",
        keep="first"
    )
    .copy()
)

print(
    "Duplicate Google Books IDs removed:",
    before - len(expansion_candidates)
)


# ------------------------------------------------------------
# Then deduplicate same title + author
# ------------------------------------------------------------

before = len(expansion_candidates)

expansion_candidates = (
    expansion_candidates
    .drop_duplicates(
        subset="_work_key",
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Duplicate title + author records removed:",
    before - len(expansion_candidates)
)


print(
    "\nUnique expansion candidates:",
    len(expansion_candidates)
)


display(
    expansion_candidates[
        [
            "title",
            "authors",
            "collection_theme",
            "discovery_query"
        ]
    ].head(20)
)

Removed records without title: 0
Duplicate Google Books IDs removed: 40
Duplicate title + author records removed: 29

Unique expansion candidates: 691


,title,authors,collection_theme,discovery_query
0,Interspecies Negotiations,"Susan Haris, Anu Pande",humans_and_nature,multispecies kinship
1,Just Like Family,Andrea Laurent-Simpson,humans_and_nature,multispecies kinship
2,Critical Terms for Animal Studies,Lori Gruen,humans_and_nature,multispecies kinship
3,Inhabitation,Gry Worre Hallberg,humans_and_nature,multispecies kinship
4,Dreams of Dawn,"Marti J. Steussy, Marti Steussy",humans_and_nature,multispecies kinship
5,Individuality Incorporated,Joel Pfister,humans_and_nature,multispecies kinship
6,Spiritual Practices for Ecological Survival,Cláudio Carvalhaes,humans_and_nature,multispecies kinship
7,Proceedings of a Conference on Multispecies Gr...,"Frank H. Baker, R. Katherine Jones",humans_and_nature,multispecies kinship
8,Multi-species North Sea Fisheries,Chungsoo Kim,humans_and_nature,multispecies kinship
9,Fieldiana,,humans_and_nature,multispecies kinship


Step 5: Remove books already represented in the 738 catalogue

In [ ]:
# ============================================================
# Step 5: Compare expansion candidates with current 738 books
# ============================================================

# Create normalized keys for the current expanded catalogue

catalogue_check = expanded_books.copy()

catalogue_title_col = (
    "title_final"
    if "title_final" in catalogue_check.columns
    else "title"
)

catalogue_author_col = (
    "author_final"
    if "author_final" in catalogue_check.columns
    else "author"
)


catalogue_check["_title_key"] = (
    catalogue_check[catalogue_title_col]
    .apply(normalize_expansion_text)
)

catalogue_check["_author_key"] = (
    catalogue_check[catalogue_author_col]
    .apply(normalize_expansion_text)
)

catalogue_check["_work_key"] = (
    catalogue_check["_title_key"]
    + " | "
    + catalogue_check["_author_key"]
)


# Existing keys

existing_work_keys = set(
    catalogue_check["_work_key"]
)

existing_title_keys = set(
    catalogue_check["_title_key"]
)


# Compare expansion candidates

expansion_candidates["same_work_existing"] = (
    expansion_candidates["_work_key"]
    .isin(existing_work_keys)
)

expansion_candidates["same_title_existing"] = (
    expansion_candidates["_title_key"]
    .isin(existing_title_keys)
)


print("Unique API candidates:", len(expansion_candidates))

print(
    "Exact title + author already present:",
    expansion_candidates[
        "same_work_existing"
    ].sum()
)

print(
    "Same title already present:",
    expansion_candidates[
        "same_title_existing"
    ].sum()
)


# Remove only definite same work duplicates

expansion_new = expansion_candidates[
    ~expansion_candidates["same_work_existing"]
].copy()

expansion_new = expansion_new.reset_index(drop=True)


print(
    "Candidates after definite duplicate removal:",
    len(expansion_new)
)

Unique API candidates: 691
Exact title + author already present: 17
Same title already present: 39
Candidates after definite duplicate removal: 674


In [ ]:
# ============================================================
# Step 6: Metadata quality screening
# ============================================================

# Clean descriptions

expansion_new["description"] = (
    expansion_new["description"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Word count

expansion_new["description_words"] = (
    expansion_new["description"]
    .str.split()
    .str.len()
)


# Basic metadata indicators

expansion_new["has_author"] = (
    expansion_new["authors"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

expansion_new["has_description"] = (
    expansion_new["description_words"] >= 15
)

expansion_new["has_isbn"] = (
    expansion_new["isbn_13"].notna()
    | expansion_new["isbn_10"].notna()
)


# Metadata quality score
# 0 to 4

expansion_new["metadata_quality"] = (
    expansion_new["has_author"].astype(int)
    + expansion_new["has_description"].astype(int)
    + expansion_new["has_isbn"].astype(int)
    + expansion_new["categories"].fillna("").ne("").astype(int)
)


print("NEW CANDIDATE QUALITY")
print("Candidates:", len(expansion_new))

print(
    "With useful descriptions:",
    expansion_new["has_description"].sum()
)

print(
    "Without useful descriptions:",
    (~expansion_new["has_description"]).sum()
)

print(
    "With ISBN:",
    expansion_new["has_isbn"].sum()
)


print("\nMetadata quality distribution")

display(
    expansion_new[
        "metadata_quality"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("quality_score")
    .reset_index(name="books")
)


print("\nDescription length")

display(
    expansion_new[
        "description_words"
    ].describe()
)

NEW CANDIDATE QUALITY
Candidates: 674
With useful descriptions: 425
Without useful descriptions: 249
With ISBN: 366

Metadata quality distribution


,quality_score,books
0,0,9
1,1,80
2,2,142
3,3,124
4,4,319



Description length


,description_words
count,674.000000
mean,108.695846
std,125.247159
min,0.000000
25%,0.000000
50%,68.000000
75%,184.000000
max,685.000000


## Step 7: Create semantic text and calculate relevance
This is where the expansion gets much stronger than the original collection.
We will embed the candidate books and compare them with a semantic description of each Kinship Library theme. Importantly, the discovery query itself is not included in the book embedding.

In [ ]:
# ============================================================
# Step 7: Semantic relevance screening
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Keep candidates with usable semantic information
# ------------------------------------------------------------

expansion_semantic = expansion_new[
    expansion_new["has_description"]
].copy().reset_index(drop=True)


# ------------------------------------------------------------
# 2. Construct semantic text
# ------------------------------------------------------------

expansion_semantic["semantic_text"] = (
    expansion_semantic["title"]
    .fillna("")
    .astype(str)
    + ". "
    + expansion_semantic["authors"]
    .fillna("")
    .astype(str)
    + ". "
    + expansion_semantic["description"]
    .fillna("")
    .astype(str)
)


print(
    "Candidates entering semantic screening:",
    len(expansion_semantic)
)


# ------------------------------------------------------------
# 3. Encode candidates
# ------------------------------------------------------------

candidate_embeddings = model.encode(
    expansion_semantic[
        "semantic_text"
    ].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)


print(
    "Candidate embeddings:",
    candidate_embeddings.shape
)


# ------------------------------------------------------------
# 4. Semantic definitions of the five collection paths
# ------------------------------------------------------------

THEME_DESCRIPTIONS = {

    "humans_and_nature":
        """
        Human relationships with Nature, ecology, more than human
        life, multispecies relations, environmental philosophy,
        ecological belonging, plants, animals, forests, living
        systems and the interdependence of humans and ecosystems.
        """,

    "indigenous_knowledge_americas":
        """
        Indigenous knowledge and philosophy of the Americas,
        especially South American, Andean, Amazonian, Aymara,
        Quechua, Kogi and Muisca thought, traditional ecological
        knowledge, relational ontology, territory, decolonial
        thought, Buen Vivir, Sumak Kawsay and Indigenous
        relationships with land and more than human life.
        """,

    "yoga_philosophy":
        """
        Classical and contemporary philosophy of Yoga including
        Patanjali, the Yoga Sutras, Bhagavad Gita, Vedanta,
        Advaita, Upanishads, Samkhya, consciousness, liberation,
        ethics, self knowledge and Indian philosophical inquiry.
        """,

    "ayurveda_indian_spirituality":
        """
        Ayurveda, Indian spirituality, Tantra, Shiva and Shakti
        traditions, Kashmir Shaivism, subtle body philosophy,
        prana, Vedic thought, sacred ecology and Indian
        understandings of body, mind, consciousness and Nature.
        """,

    "yoga_way_of_life":
        """
        Yoga as an embodied way of life including asana,
        pranayama, meditation, sadhana, self inquiry, ethics,
        contemplative practice, daily spiritual practice and
        integration of Yoga into everyday life.
        """
}


theme_names = list(
    THEME_DESCRIPTIONS.keys()
)


theme_embeddings = model.encode(
    [
        THEME_DESCRIPTIONS[theme]
        for theme in theme_names
    ],
    normalize_embeddings=True
)


# ------------------------------------------------------------
# 5. Similarity with every theme
# ------------------------------------------------------------

similarities = (
    candidate_embeddings
    @ theme_embeddings.T
)


expansion_semantic["semantic_theme"] = [
    theme_names[index]
    for index in np.argmax(
        similarities,
        axis=1
    )
]


expansion_semantic["semantic_relevance"] = (
    np.max(
        similarities,
        axis=1
    )
)


# Similarity to its original discovery route theme

theme_index = {
    theme: index
    for index, theme
    in enumerate(theme_names)
}


expansion_semantic[
    "discovery_theme_similarity"
] = [

    similarities[row_index][
        theme_index[theme]
    ]

    for row_index, theme
    in enumerate(
        expansion_semantic[
            "collection_theme"
        ]
    )
]


# Does semantic classification agree
# with the discovery route?

expansion_semantic[
    "theme_agreement"
] = (
    expansion_semantic[
        "semantic_theme"
    ]
    ==
    expansion_semantic[
        "collection_theme"
    ]
)


print("\nSEMANTIC SCREENING COMPLETE")

print(
    "Books evaluated:",
    len(expansion_semantic)
)

print(
    "Mean semantic relevance:",
    round(
        expansion_semantic[
            "semantic_relevance"
        ].mean(),
        3
    )
)

print(
    "Median semantic relevance:",
    round(
        expansion_semantic[
            "semantic_relevance"
        ].median(),
        3
    )
)

print(
    "Discovery theme agreement:",
    round(
        expansion_semantic[
            "theme_agreement"
        ].mean() * 100,
        1
    ),
    "%"
)


display(
    expansion_semantic[
        [
            "title",
            "authors",
            "collection_theme",
            "semantic_theme",
            "semantic_relevance",
            "theme_agreement"
        ]
    ]
    .sort_values(
        "semantic_relevance",
        ascending=False
    )
    .head(20)
)

Candidates entering semantic screening: 425


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Candidate embeddings: (425, 384)

SEMANTIC SCREENING COMPLETE
Books evaluated: 425
Mean semantic relevance: 0.444
Median semantic relevance: 0.457
Discovery theme agreement: 69.9 %


,title,authors,collection_theme,semantic_theme,semantic_relevance,theme_agreement
228,The Yoga Sutras of Patanjali: The Book of the ...,Patañjali,yoga_philosophy,yoga_philosophy,0.780784,True
223,Yoga Philosophy of Patañjali,"Patañjali, Swāmi Āraṇya Hariharānanda",yoga_philosophy,yoga_philosophy,0.779078,True
272,Yoga Philosophy of Patañjali,Swāmi Āraṇya Hariharānanda,yoga_philosophy,yoga_philosophy,0.774946,True
266,CLASSICAL YOGA PHILOSOPHY,Dr. Naveen Kaushik,yoga_philosophy,yoga_philosophy,0.774920,True
226,The Yoga Sutras of Patanjali,By Patanjali,yoga_philosophy,yoga_philosophy,0.765590,True
273,The Encyclopedia of Indian Philosophies,"Karl H. Potter, Gerald James Larson",yoga_philosophy,yoga_philosophy,0.762345,True
270,The Philosophy of the Yogasutra,Karen O'Brien-Kop,yoga_philosophy,yoga_philosophy,0.752078,True
229,Yoga Sutras of Patanjali,B. Ravikanth,yoga_philosophy,yoga_philosophy,0.746952,True
222,An Introduction to Yoga Philosophy,Ashok Kumar Malhotra,yoga_philosophy,yoga_philosophy,0.744704,True
225,PATANJALI'S YOGA SUTRA,Shyam Ranganathan,yoga_philosophy,yoga_philosophy,0.744328,True


In [ ]:
display(
    expansion_semantic[
        "semantic_relevance"
    ].describe(
        percentiles=[
            .10,
            .20,
            .25,
            .50,
            .75,
            .90
        ]
    )
)

display(
    expansion_semantic.groupby(
        "collection_theme"
    ).agg(
        candidates=("title", "count"),
        mean_relevance=(
            "semantic_relevance",
            "mean"
        ),
        median_relevance=(
            "semantic_relevance",
            "median"
        ),
        theme_agreement=(
            "theme_agreement",
            "mean"
        )
    ).round(3)
)

,semantic_relevance
count,425.000000
mean,0.444492
std,0.141374
min,0.088446
10%,0.255469
20%,0.304137
25%,0.334330
50%,0.456804
75%,0.544635
90%,0.622300


,candidates,mean_relevance,median_relevance,theme_agreement
collection_theme,,,,
ayurveda_indian_spirituality,72,0.503,0.509,0.750
humans_and_nature,92,0.383,0.385,0.815
indigenous_knowledge_americas,130,0.389,0.392,0.715
yoga_philosophy,57,0.573,0.558,0.632
yoga_way_of_life,74,0.462,0.465,0.527


## 20.8 Step 8: Theme aware final selection
Let's retain approximately 93% of each theme, which gives us about 395 books. Within each discovery theme, the books with the lowest semantic relevance to any Kinship theme are removed.

In [ ]:
# ============================================================
# Step 8 revised
# Remove weakest semantic tail before final expansion
# ============================================================

MIN_RELEVANCE = 0.25


expansion_quality = expansion_semantic[
    expansion_semantic["semantic_relevance"]
    >= MIN_RELEVANCE
].copy()


print("Before quality floor:", len(expansion_semantic))
print("After quality floor:", len(expansion_quality))
print(
    "Removed:",
    len(expansion_semantic) - len(expansion_quality)
)


print("\nBOOKS REMOVED BY THEME")

display(
    expansion_semantic[
        expansion_semantic["semantic_relevance"]
        < MIN_RELEVANCE
    ]["collection_theme"]
    .value_counts()
    .rename_axis("collection_theme")
    .reset_index(name="removed")
)


print("\nBOOKS RETAINED BY THEME")

display(
    expansion_quality[
        "collection_theme"
    ]
    .value_counts()
    .rename_axis("collection_theme")
    .reset_index(name="retained")
)

Before quality floor: 425
After quality floor: 387
Removed: 38

BOOKS REMOVED BY THEME


,collection_theme,removed
0,humans_and_nature,16
1,indigenous_knowledge_americas,16
2,yoga_way_of_life,4
3,ayurveda_indian_spirituality,2



BOOKS RETAINED BY THEME


,collection_theme,retained
0,indigenous_knowledge_americas,114
1,humans_and_nature,76
2,ayurveda_indian_spirituality,70
3,yoga_way_of_life,70
4,yoga_philosophy,57


In [ ]:
print(
    "Final candidate pool:",
    len(expansion_quality)
)

print(
    "Minimum relevance:",
    round(
        expansion_quality[
            "semantic_relevance"
        ].min(),
        3
    )
)

print(
    "Mean relevance:",
    round(
        expansion_quality[
            "semantic_relevance"
        ].mean(),
        3
    )
)


display(
    expansion_quality[
        [
            "title",
            "authors",
            "collection_theme",
            "semantic_theme",
            "semantic_relevance",
            "discovery_query",
            "description_words"
        ]
    ]
    .sort_values(
        "semantic_relevance"
    )
    .head(20)
)

Final candidate pool: 387
Minimum relevance: 0.251
Mean relevance: 0.468


,title,authors,collection_theme,semantic_theme,semantic_relevance,discovery_query,description_words
83,Designing Regenerative Cultures,Daniel Wahl,humans_and_nature,indigenous_knowledge_americas,0.251201,regenerative ecology,71
57,Philosophical Transactions of the Royal Societ...,,humans_and_nature,humans_and_nature,0.251933,forest ecology philosophy,44
401,10 Last Years Solved Papers Commerce Stream : ...,Oswal,yoga_way_of_life,ayurveda_indian_spirituality,0.253175,Yoga and daily life,215
97,The ^AOxford Handbook of Feminist Philosophy,"Ásta, Kim Q. Hall",indigenous_knowledge_americas,yoga_philosophy,0.253225,Indigenous philosophy South America,84
117,Genes in the Field,International Plant Genetic Resources Institut...,indigenous_knowledge_americas,humans_and_nature,0.254048,Andean cosmology,155
193,Post-Apocalyptic Cultures,"Julia Urabayen, Jorge León Casero",indigenous_knowledge_americas,indigenous_knowledge_americas,0.257601,Sumak Kawsay,142
28,Green Consumption in China,Jin Min,humans_and_nature,humans_and_nature,0.257806,ecological philosophy,166
123,The Routledge Handbook of Transformative Globa...,"S. A. Hamed Hosseini, James Goodman, Sara C. M...",indigenous_knowledge_americas,indigenous_knowledge_americas,0.259760,Amazonian Indigenous philosophy,222
7,Energy Democracies for Sustainable Futures,"Majia Nadesan, Martin J. Pasqualetti, Jennifer...",humans_and_nature,humans_and_nature,0.260330,more than human ecology,235
131,The SAGE Handbook of Qualitative Business and ...,"Catherine Cassell, Ann L Cunliffe, Gina Grandy",indigenous_knowledge_americas,yoga_philosophy,0.260340,Indigenous relational ontology,298


In [ ]:
# ============================================================
# Step 10: Remove obvious catalogue noise
# ============================================================

import re


NOISE_PATTERNS = [
    r"\bsolved papers?\b",
    r"\bquestion papers?\b",
    r"\bexam\b",
    r"\bexamination\b",
    r"\btest prep\b",
    r"\bpractice tests?\b",
    r"\bjournal of\b",
    r"\bproceedings\b",
    r"\bannual report\b",
    r"\bconference proceedings\b",
]


noise_regex = "|".join(NOISE_PATTERNS)


expansion_quality["noise_flag"] = (
    expansion_quality["title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains(
        noise_regex,
        regex=True
    )
)


print(
    "Potential obvious noise:",
    expansion_quality["noise_flag"].sum()
)


print("\nREMOVED AS OBVIOUS NOISE")

display(
    expansion_quality[
        expansion_quality["noise_flag"]
    ][
        [
            "title",
            "authors",
            "collection_theme",
            "discovery_query",
            "semantic_relevance"
        ]
    ]
)


# Keep the clean candidate pool

expansion_final = expansion_quality[
    ~expansion_quality["noise_flag"]
].copy().reset_index(drop=True)


print(
    "\nFinal expansion books:",
    len(expansion_final)
)

Potential obvious noise: 4

REMOVED AS OBVIOUS NOISE


,title,authors,collection_theme,discovery_query,semantic_relevance
400,10 Last Years Solved Papers Humanities Stream ...,Oswal,yoga_way_of_life,Yoga and daily life,0.316576
401,10 Last Years Solved Papers Commerce Stream : ...,Oswal,yoga_way_of_life,Yoga and daily life,0.253175
402,10 Last Years Solved Papers for Science (PCM) ...,Oswal,yoga_way_of_life,Yoga and daily life,0.296145
403,10 Last Years Solved Papers for Science (PCB) ...,Oswal,yoga_way_of_life,Yoga and daily life,0.295551



Final expansion books: 383


In [ ]:
display(
    expansion_final[
        [
            "title",
            "authors",
            "collection_theme",
            "semantic_theme",
            "semantic_relevance",
            "discovery_query"
        ]
    ]
    .sort_values("semantic_relevance")
    .head(20)
)

,title,authors,collection_theme,semantic_theme,semantic_relevance,discovery_query
71,Designing Regenerative Cultures,Daniel Wahl,humans_and_nature,indigenous_knowledge_americas,0.251201,regenerative ecology
51,Philosophical Transactions of the Royal Societ...,,humans_and_nature,humans_and_nature,0.251933,forest ecology philosophy
79,The ^AOxford Handbook of Feminist Philosophy,"Ásta, Kim Q. Hall",indigenous_knowledge_americas,yoga_philosophy,0.253225,Indigenous philosophy South America
98,Genes in the Field,International Plant Genetic Resources Institut...,indigenous_knowledge_americas,humans_and_nature,0.254048,Andean cosmology
165,Post-Apocalyptic Cultures,"Julia Urabayen, Jorge León Casero",indigenous_knowledge_americas,indigenous_knowledge_americas,0.257601,Sumak Kawsay
25,Green Consumption in China,Jin Min,humans_and_nature,humans_and_nature,0.257806,ecological philosophy
103,The Routledge Handbook of Transformative Globa...,"S. A. Hamed Hosseini, James Goodman, Sara C. M...",indigenous_knowledge_americas,indigenous_knowledge_americas,0.259760,Amazonian Indigenous philosophy
6,Energy Democracies for Sustainable Futures,"Majia Nadesan, Martin J. Pasqualetti, Jennifer...",humans_and_nature,humans_and_nature,0.260330,more than human ecology
110,The SAGE Handbook of Qualitative Business and ...,"Catherine Cassell, Ann L Cunliffe, Gina Grandy",indigenous_knowledge_americas,yoga_philosophy,0.260340,Indigenous relational ontology
3,Individuality Incorporated,Joel Pfister,humans_and_nature,indigenous_knowledge_americas,0.260539,multispecies kinship


### Preserving the catalogue to add it as feature in the final app


In [ ]:
# ============================================================
# Preserve catalogue provenance before final merge
# ============================================================

# Existing 713 catalogue
app_books["is_curated"] = False
app_books["catalogue_source"] = "original_collection"

# 25 curated books
curated_for_app["is_curated"] = True
curated_for_app["catalogue_source"] = "creator_curated"

# 383 broader expansion books
expansion_final["is_curated"] = False
expansion_final["catalogue_source"] = "semantic_expansion"


print("Original:", len(app_books))
print("Curated:", len(curated_for_app))
print("Expansion:", len(expansion_final))

print(
    "Physical rows after final merge:",
    len(app_books)
    + len(curated_for_app)
    + len(expansion_final)
)

Original: 713
Curated: 25
Expansion: 383
Physical rows after final merge: 1121


### 23. SECTION:
Step 11: align the 383 expansion books with their embeddings
Because expansion_semantic and candidate_embeddings were created in exactly the same order, we can preserve an embedding index and select safely:

In [ ]:
# ============================================================
# Step 11: Align final expansion with candidate embeddings
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Give every semantic candidate its embedding position
# ------------------------------------------------------------

expansion_semantic = (
    expansion_semantic
    .reset_index(drop=True)
    .copy()
)

expansion_semantic["embedding_index"] = (
    np.arange(len(expansion_semantic))
)


# ------------------------------------------------------------
# 2. Recover embedding positions for the final 383 books
# ------------------------------------------------------------

embedding_lookup = (
    expansion_semantic[
        [
            "google_books_id",
            "embedding_index"
        ]
    ]
    .drop_duplicates("google_books_id")
)


expansion_final_aligned = (
    expansion_final
    .merge(
        embedding_lookup,
        on="google_books_id",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 3. Safety checks
# ------------------------------------------------------------

assert expansion_final_aligned[
    "embedding_index"
].notna().all()

expansion_indices = (
    expansion_final_aligned[
        "embedding_index"
    ]
    .astype(int)
    .to_numpy()
)


expansion_final_embeddings = (
    candidate_embeddings[
        expansion_indices
    ]
)


assert len(expansion_final_aligned) == 383

assert expansion_final_embeddings.shape == (
    383,
    384
)


print(
    "Final expansion books:",
    len(expansion_final_aligned)
)

print(
    "Final expansion embeddings:",
    expansion_final_embeddings.shape
)

print(
    "Expansion alignment:",
    len(expansion_final_aligned)
    == len(expansion_final_embeddings)
)

Final expansion books: 383
Final expansion embeddings: (383, 384)
Expansion alignment: True


### 24. Step 12: build the final 1,121 book catalogue
Now we combine your already aligned 738 book curated checkpoint with these 383.

In [4]:
# ============================================================
# Step 12: Final catalogue and embedding merge
# ============================================================

# Make sure provenance exists in the 738 book catalogue

expanded_books["is_curated"] = False
expanded_books["catalogue_source"] = "original_collection"

# Last 25 rows are the curated addition
expanded_books.loc[
    expanded_books.index[-25:],
    "is_curated"
] = True

expanded_books.loc[
    expanded_books.index[-25:],
    "catalogue_source"
] = "creator_curated"


# ------------------------------------------------------------
# Prepare expansion rows for app structure
# ------------------------------------------------------------

expansion_final_aligned["title_final"] = (
    expansion_final_aligned["title"]
)

expansion_final_aligned["author_final"] = (
    expansion_final_aligned["authors"]
)

expansion_final_aligned["description_final"] = (
    expansion_final_aligned["description"]
)

expansion_final_aligned["is_curated"] = False

expansion_final_aligned["catalogue_source"] = (
    "semantic_expansion"
)


# Add missing app columns

for col in expanded_books.columns:
    if col not in expansion_final_aligned.columns:
        expansion_final_aligned[col] = pd.NA


expansion_for_app = (
    expansion_final_aligned[
        expanded_books.columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Final merge
# ------------------------------------------------------------

final_books = pd.concat(
    [
        expanded_books.reset_index(drop=True),
        expansion_for_app.reset_index(drop=True)
    ],
    ignore_index=True
)


final_embeddings = np.vstack(
    [
        expanded_embeddings,
        expansion_final_embeddings
    ]
)


# ------------------------------------------------------------
# Critical final checks
# ------------------------------------------------------------

assert len(final_books) == 1121
assert final_embeddings.shape == (1121, 384)
assert len(final_books) == len(final_embeddings)


print("\nFINAL KINSHIP LIBRARY")
print("Books:", len(final_books))
print("Embeddings:", final_embeddings.shape)

print(
    "Curated books:",
    final_books["is_curated"].sum()
)

print("\nCatalogue sources:")
print(
    final_books[
        "catalogue_source"
    ].value_counts()
)

print(
    "\nCatalogue and embeddings aligned:",
    len(final_books)
    == len(final_embeddings)
)

NameError: name 'expanded_books' is not defined

## 25. Save final production files



In [2]:
# ============================================================
# Step 13: Save final Kinship Library production files
# ============================================================

import os
import numpy as np


PROCESSED_DIR = (
    "/content/drive/MyDrive/kinship_library/"
    "data/processed"
)

MODELS_DIR = (
    "/content/drive/MyDrive/kinship_library/"
    "models"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


# ------------------------------------------------------------
# Final safety check
# ------------------------------------------------------------

assert len(final_books) == 1121
assert final_embeddings.shape == (1121, 384)
assert len(final_books) == len(final_embeddings)
assert final_books["is_curated"].sum() == 25


# ------------------------------------------------------------
# Save production catalogue
# ------------------------------------------------------------

FINAL_BOOKS_PATH = (
    f"{PROCESSED_DIR}/app_books_expanded.csv"
)

final_books.to_csv(
    FINAL_BOOKS_PATH,
    index=False
)


# ------------------------------------------------------------
# Save production embeddings
# ------------------------------------------------------------

FINAL_EMBEDDINGS_PATH = (
    f"{MODELS_DIR}/app_book_embeddings_expanded.npy"
)

np.save(
    FINAL_EMBEDDINGS_PATH,
    final_embeddings
)


print("FINAL PRODUCTION FILES SAVED")
print()
print("Books:")
print(FINAL_BOOKS_PATH)
print()
print("Embeddings:")
print(FINAL_EMBEDDINGS_PATH)
print()
print("Books:", len(final_books))
print("Embeddings:", final_embeddings.shape)
print(
    "Curated:",
    final_books["is_curated"].sum()
)

NameError: name 'final_books' is not defined

In [66]:
# ============================================================
# Step 14: Reload and verify saved production files
# ============================================================

saved_books = pd.read_csv(
    FINAL_BOOKS_PATH
)

saved_embeddings = np.load(
    FINAL_EMBEDDINGS_PATH
)


print("RELOAD CHECK")
print("Books:", len(saved_books))
print("Embeddings:", saved_embeddings.shape)

print(
    "Curated:",
    saved_books["is_curated"].sum()
)

print(
    "Aligned:",
    len(saved_books)
    == len(saved_embeddings)
)

print("\nSources:")
print(
    saved_books[
        "catalogue_source"
    ].value_counts()
ing )

RELOAD CHECK
Books: 1121
Embeddings: (1121, 384)
Curated: 25
Aligned: True

Sources:
catalogue_source
original_collection    713
semantic_expansion     383
creator_curated         25
Name: count, dtype: int64


## Final checkpoint

At the end of this notebook you should have:

1. A transparent audit of curated foundational books.
2. A place where you can add your own titles.
3. Metadata for missing curated books.
4. New embeddings made with the same MiniLM model.
5. Safe expanded files that do not overwrite the working Streamlit catalogue.
6. A set of broader expansion queries ready for a second collection pass toward 1000 plus books.